# M5 · Document worker: line items from M4 page readings

This notebook trains the **M5 line-item model**. M5 takes the text that M4 (page reading) found on a workshop estimate and says what each piece of text *is*: a part description, an operation, a quantity, a printed amount, or something else. The model is **LayoutLMv3**, a transformer that reads text, the position of the text on the page, and the page image together.

**What this notebook does**

1. Loads the output of the real M4 module on the 500 synthetic pages of `data/raw/synthetic_motor_claims_500_w_annotation` (the *M4 cache*).
2. Gives every M4 text box a training label by comparing it with the dataset's ground-truth field boxes, and shows where that is impossible.
3. Fine-tunes LayoutLMv3 on the training split and selects the best epoch on the validation split.
4. Scores the result at three levels: text boxes, table rows, and finished `LineItem` records produced by the application's own serving code.
5. Writes down the row and amount boxes that M6 (pen marks) receives from these line items.
6. Exports a **candidate** registry entry that `src/claim_cmev/documents/line_items/layoutlmv3.py` can load.

**What you will learn**

- What M4 hands to M5 (the `DocumentPage` record) and the five coordinate frames a text box passes through.
- Why training labels must be built on M4's own output, and how a box is labelled, marked `O`, or left out.
- How LayoutLMv3 turns a page into tensors, what the tensors mean, and how the loss ignores unlabelled positions.
- How to read learning curves, per-class scores, confusion matrices, a reliability plot and row-level results.
- Why *finding the amount box* and *reading the amount correctly* are two different problems.

> **Status (honest box).** Everything here is **synthetic validation evidence**: one page layout, three handwriting *fonts*, no photograph, no real estimate. A score in this notebook says how the model labels pages like these. It does not say how it reads a real estimate, and it does not show that any project acceptance target is reached. The exported model is a `candidate`. Thresholds, the label policy and the recipe are **proposed** until the team accepts them. The held-out test split is never read unless you set `FINAL_EVAL = True`.
>
> **Three results to read together.** In the reference run of 2026-10-12 the model labelled every scored validation box correctly (4,324 of 4,324). A lookup that uses only the four coordinates of each box, with no text, no image and no model, labels 4,285 of the same 4,324 boxes correctly (section 8.2). So the task on this generator is easy, and the score mostly measures its fixed layout. The third result is the stress probe of section 8.9, which moves the same validation pages. The row numbers survive a move of 10 pixels in any direction. They break at a fraction of a row height, up or down: 20 pixels down already costs 83 of 716 table cells, and a move of one row height (79 pixels) leaves 19 of 716 cells right when the page is moved down and 162 of 716 when it is moved up, with all the other cells off by exactly one row. The field type (description, amount, …) stays right for at least 715 of 716 cells under every vertical move tested, and is lost for 22 to 28 of 716 cells under a sideways move of 40 pixels. So this model takes the row number from the absolute height on the page, and it is not ready for a page whose table sits elsewhere. Section 13 lists the reference numbers.

**How this relates to `notebooks/Document_Worker.ipynb`.** That notebook, written by a teammate, was the first LayoutLMv3 experiment in this project and it produced the checkpoint in `artifacts/models/document-worker-layoutlmv3`. It trained on the dataset's *annotation* text and boxes: each ground-truth field value was one input "word". That was a sensible first step and its checkpoint loads and runs. It has one structural gap, which this notebook closes: the application never sees annotation boxes. It sees **M4's OCR output**, which also contains form labels ("Claim Number"), column headers, handwriting and OCR mistakes, none of which that model was shown, and none of which it could label `O` because no training example was `O`. Here the model is trained on exactly what M4 delivers.

## Where M5 sits

```mermaid
flowchart LR
    U[Uploaded estimate page] --> M4[M4 page reading<br/>pretrained OCR, no training]
    M4 -->|DocumentPage: text boxes + corrected image| M5[M5 line items<br/>THIS NOTEBOOK]
    M4 -->|corrected image| M6[M6 pen marks<br/>detector]
    M5 -->|rows and amount boxes| M6
    M5 -->|LineItem records + completeness| M8[M8 checks<br/>deterministic rules]
    M6 -->|pending marks| M8
    M8 --> M9[M9 review by the surveyor]
```

M4 is not trained here. It is pretrained OCR that the project serves as it is. M5 must therefore learn to work with whatever M4 produces, including its mistakes. M6 detects surveyor pen marks and needs M5's rows to link a mark to a row. M8 applies fixed rules; no model decides a finding. The next cell draws the same pipeline with matplotlib, for viewers that do not render Mermaid.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

def pipeline_figure():
    fig, ax = plt.subplots(figsize=(11, 2.6))
    stages = [("Uploaded\npage", "#DDDDDD"), ("M4\npage reading\n(pretrained OCR)", "#56B4E9"),
              ("M5\nline items\n(this notebook)", "#E69F00"), ("M6\npen marks", "#56B4E9"),
              ("M8\nchecks\n(rules)", "#DDDDDD"), ("M9\nsurveyor\nreview", "#DDDDDD")]
    for i, (name, colour) in enumerate(stages):
        ax.add_patch(FancyBboxPatch((i * 1.8, 0.2), 1.3, 1.0, boxstyle="round,pad=0.04", facecolor=colour, edgecolor="black"))
        ax.text(i * 1.8 + 0.65, 0.7, name, ha="center", va="center", fontsize=9)
        if i:
            ax.annotate("", xy=(i * 1.8 - 0.04, 0.7), xytext=(i * 1.8 - 0.46, 0.7), arrowprops={"arrowstyle": "->"})
    ax.annotate("rows and amount boxes", xy=(3 * 1.8 + 0.65, 1.24), xytext=(2 * 1.8 + 0.65, 1.62), fontsize=8, ha="center",
                arrowprops={"arrowstyle": "->", "connectionstyle": "arc3,rad=-0.25"})
    ax.text(1 * 1.8 + 1.55, 0.02, "DocumentPage:\ntext boxes + corrected image", fontsize=8, ha="center", va="top")
    ax.set_xlim(-0.2, 10.6); ax.set_ylim(-0.45, 1.9); ax.axis("off")
    ax.set_title("Document branch of CLAIM-CMEV: M5 labels the text M4 found")
    return fig

pipeline_fig = pipeline_figure()
plt.show()

## 1 · Environment and paths

Use the project environment as the Jupyter kernel (`CLAIM-CMEV (Python 3.12)`). The first cell finds the repository root and points the Hugging Face and PyTorch caches at `artifacts/models/.cache/`, **before** `torch` or `transformers` is imported, because those libraries read the cache location once at import time.

The base checkpoint `microsoft/layoutlmv3-base` is loaded from that local cache without touching the network. If it is not there yet, set `OFFLINE = False` in the configuration cell for one run.

In [ ]:
from pathlib import Path
import os
import sys

# Works when Jupyter starts in the repository root or in a subfolder such as notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "docs/specs/README.md").is_file() and (p / "pipelines").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the CLAIM-CMEV checkout, or set ROOT to its path.")
for entry in (str(ROOT), str(ROOT / "src")):
    if entry not in sys.path:
        sys.path.insert(0, entry)
# Set caches before importing transformers/torch; keep any value the user already set.
os.environ.setdefault("HF_HOME", str(ROOT / "artifacts/models/.cache/huggingface"))
os.environ.setdefault("TORCH_HOME", str(ROOT / "artifacts/models/.cache/torch"))
print("Repository:", ROOT)
print("Kernel Python:", sys.executable)

The install cell is **off**. Turn it on only to install the training dependencies into this kernel's environment, then restart the kernel and run the notebook again from the top.

In [ ]:
INSTALL_DEPENDENCIES = False  # True installs into this kernel's environment; restart the kernel afterwards.
if INSTALL_DEPENDENCIES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r",
                           str(ROOT / "requirements-training.txt"), "-e", str(ROOT) + "[vision]"])
    print("Restart the kernel after installation, then rerun this notebook.")

The imports below bring in three helper modules written for this notebook. They hold the mechanics that deserve unit tests; the notebook shows you what they do.

| Module | What it holds |
| --- | --- |
| `pipelines/documents/lineitems_data.py` | the label policy, the box-to-field alignment, OCR text quality, the window dataset |
| `pipelines/documents/lineitems_training.py` | the model builder, the training loop, inference, reload, export, the guards behind the switches, the rows handed to M6 |
| `pipelines/documents/lineitems_eval.py` | every score with its denominator, the position-only baseline, and the shared plot helpers |
| `src/claim_cmev/documents/line_items/layoutlmv3.py` | the **serving adapter**. Training imports its `page_words` and `encode_page`, so both sides build identical tensors |

In [ ]:
import inspect
import json
import math
import shutil
import tempfile
import time
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import torch
import transformers
from IPython.display import display
from PIL import Image

from claim_cmev.contracts.common import Provenance, make_job_key
from claim_cmev.documents.line_items import load_estimate_vocabulary, load_layout_families
from claim_cmev.documents.line_items.layoutlmv3 import (
    LocalLayoutLMv3, PageWords, adapt_predictions, encode_page, layout_bbox, page_words, serving_preprocessing)
from claim_cmev.documents.line_items.parser import pen_mark_row_boxes
from claim_cmev.vision.registry import sha256_file
from pipelines.documents.m4_cache import (
    M4CacheError, corrected_boxes_to_norm, load_m4_cache, source_to_corrected_boxes)
from pipelines.documents.lineitems_data import (
    IGNORE_INDEX, LabelPolicy, PageWindowDataset, align_page, align_pages, assert_claims_in_one_split, bio_labels,
    box_rows as alignment_box_rows, collate_windows, field_name, field_rows, field_type, overlap_of_smaller,
    overlap_samples, read_annotations, read_claims, region_label_names, row_slot, share_inside, text_quality_rows)
from pipelines.documents.lineitems_eval import (
    CORE_ROW_FIELDS, PALETTE, TYPE_COLOURS, TYPE_ORDER, box_rows, box_summary, draw_boxes, entity_scores,
    expected_calibration_error, label_metrics, line_item_rows, line_item_summary, plot_confusion, position_baseline,
    reliability, row_rows, row_summary, slot_confusion, style_axes, type_confusion, type_metrics, validation_report)
from pipelines.documents.lineitems_training import (
    BASE_CHECKPOINT, BASE_LICENCE, BASE_REVISION, MODEL_INPUTS, TrainConfig, build_model, describe_training_data,
    entry_problems, evaluate_pages, evaluation_folder_name, export_for_serving, load_run, lr_factor,
    masked_cross_entropy, new_run_id, open_final_evaluation, parameter_blocks, registry_entry_with_weights,
    reload_mismatches, rows_for_m6, run_inference, set_seed, train, write_json_atomic)
from pipelines.vision.training import select_device

pd.set_option("display.max_colwidth", 60, "display.width", 160)
transformers.logging.set_verbosity_error()        # the notebook prints what matters itself
transformers.logging.disable_progress_bar()
print("Python      :", sys.version.split()[0])
print("PyTorch     :", torch.__version__)
print("transformers:", transformers.__version__)
print("NumPy       :", np.__version__, "| pandas", pd.__version__)
print("CUDA available:", torch.cuda.is_available(), "| Apple MPS available:", torch.backends.mps.is_available())

## 2 · Configuration: the one cell you edit

Everything you may want to change is in the next cell.

| Switch | Default | Meaning |
| --- | --- | --- |
| `QUICK_RUN` | `False` | `True` trains on a few pages for two epochs, to check the plumbing in minutes. **Its numbers are not a model result.** |
| `FINAL_EVAL` | `False` | `True` opens the held-out test split once, at the very end. Leave it off while you are still choosing anything. It is refused in a quick run, and refused when this run already has a test report. |
| `EXPORT_FOR_SERVING` | `False` | `True` writes a `candidate` registry entry under `REGISTRY_ROOT`. It never replaces an existing entry. |
| `EXPORT_NAME` | `None` | Name of that entry. `None` means `<run_id>-serving`. To export the same run again, give a new name such as `<run_id>-serving-r2`. |
| `DEVICE` | `None` | `None` picks CUDA, then Apple MPS, then CPU. Or name one: `"cuda"`, `"mps"`, `"cpu"`. |
| `RELOAD_RUN_DIR` | `None` | Path of a saved run: skip training and evaluate that run instead. The run must have been trained with the present label policy and M4 cache; section 3.1 checks it. |

**The recipe** follows section 7.5 of `docs/specs/model_training_specification.md`. Every value is **proposed**.

| Item | Value here | Specification 7.5 | Note |
| --- | --- | --- | --- |
| Base model | `microsoft/layoutlmv3-base`, pinned revision | same | licence `cc-by-nc-sa-4.0`: non-commercial |
| Stages | one stage, on this dataset | CORD first, then generated estimates | CORD adaptation is out of scope of this notebook |
| Optimiser | AdamW, weight decay 0.01 | same | biases and normalisation weights are not decayed |
| Learning rate | 5e-5, linear warmup over 10 % of updates, then linear decay | same | |
| Batch | 4 windows × 2 accumulation steps = 8 per update | 8 | approximately the gradient of a batch of 8, with half the memory, because the GPU is shared |
| Epochs | up to 20, stop after 5 epochs without improvement | 20, early stopping | patience 5 is a choice of this notebook |
| Selection | best validation **entity F1** | same | defined below, read with the scores in section 8.2 |
| Augmentation | none | none on tokens | |
| Seed | 20260922 | same | a GPU run still differs slightly between repeats |
| Text unit | one **M4 text box** (a line) | "tokens" | the central design decision, explained in section 4 |

**Words used in this table.** Each comes back where it is used, in sections 6 to 8.

- **AdamW**: the *optimiser*, the rule that turns gradients into weight changes. It keeps a running average of each weight's gradient and of its squared gradient, and scales each weight's step by them, so every weight gets its own step size.
- **Weight decay** (0.01): at every update each weight is also shrunk by a small fraction of itself. It is a mild pull toward zero that discourages a few very large weights. The "W" in AdamW names the way this shrinking is applied.
- **Warmup and decay**: the learning rate first rises, then falls. Section 7 draws it.
- **Gradient accumulation**: the gradients of two batches of 4 are added before one weight update. This is *approximately* a batch of 8. Approximately, because each batch of 4 contributes the mean loss over its own labelled boxes: the two halves count equally even when one holds more labelled boxes than the other, while a true batch of 8 would weight every labelled box equally.
- **Gradient clipping** (`max_grad_norm = 1.0`): when the length (norm) of the whole gradient exceeds 1, it is scaled down to 1 before the update, so one unusual batch cannot throw the weights far.

**Concept: entity F1, the score that selects the checkpoint.** An *entity* is one ground-truth field together with **all** the boxes that carry it. A predicted entity counts as right only when its field label and its set of boxes are both exactly the true ones. Worked example: OCR split a description into two lines, so the field `part_2_description` has two boxes, box 7 (`B-part_2_description`) and box 8 (`I-part_2_description`). The model labels box 7 correctly and calls box 8 `O`. Box accuracy on this field is 1 of 2. The entity is **wrong**: the model's `part_2_description` is {7}, the truth is {7, 8}. Entity precision is right entities over predicted entities, recall is right entities over true entities, and F1 is `2 × right / (predicted + true)`.

Be clear about what this score can show **on this dataset**. The validation split has no field carried by two boxes (section 8.2 prints the count, and the training split has three). With one box per field, an entity is one box, so here entity F1 is in effect the accuracy on field boxes under another name. How the model handles a field of several boxes is not measured by anything in this notebook.

In [ ]:
# ----------------------------------------------------------------------------- switches
QUICK_RUN = False            # True: a few pages, two epochs; plumbing check only, NOT a model result
FINAL_EVAL = False           # True: open the held-out test split once, after everything is frozen
EXPORT_FOR_SERVING = False   # True: write a candidate registry entry (never overwrites)
EXPORT_NAME = None           # None = "<run_id>-serving"; name another entry (e.g. "<run_id>-serving-r2") to export a run again
DEVICE = None                # None = auto (CUDA, then Apple MPS, then CPU); or "cuda", "mps", "cpu"
RELOAD_RUN_DIR = None        # e.g. ROOT / "artifacts/models/lineitems/<run_id>" to skip training

# ----------------------------------------------------------------------------- paths
DATASET_DIR = ROOT / "data/raw/synthetic_motor_claims_500_w_annotation"
CACHE_DIR = ROOT / "data/processed/synthetic_motor_claims_500_w_annotation/m4-v1"
CACHE_RECEIPT = ROOT / "data/manifests/synthetic_motor_claims_m4_cache_20261012.json"
ARTIFACTS_ROOT = ROOT / "artifacts"            # runs go to artifacts/models/lineitems, reports to artifacts/evaluation
REGISTRY_ROOT = ROOT / "artifacts/models"      # where an export for serving is written
TEAMMATE_CHECKPOINT = ROOT / "artifacts/models/document-worker-layoutlmv3"

# ----------------------------------------------------------------------------- label policy (proposed)
POLICY = LabelPolicy(field_overlap_min=0.5, handwriting_overlap_min=0.5)

# ----------------------------------------------------------------------------- recipe (proposed, specification 7.5)
SEED = 20260922
OFFLINE = True               # load the base checkpoint from the local cache only
RECIPE = dict(learning_rate=5e-5, weight_decay=0.01, warmup_fraction=0.10, batch_size=4, accumulation_steps=2,
              epochs=20, patience=5, max_grad_norm=1.0, eval_batch_size=8)
QUICK_PAGES = {"train": 24, "validation": 12}  # pages per split in a quick run (whole claims)
QUICK_EPOCHS = 2

# ----------------------------------------------------------------------------- examples used in the pictures
EXAMPLE_PAGE = "SYN-00001_P2"   # a TRAINING estimate page; every picture before training uses training pages
MIN_CONFIDENCE = 0.5            # adapt_predictions default: below this a field is marked uncertain (proposed)

if QUICK_RUN:
    print("QUICK RUN: a few pages and", QUICK_EPOCHS, "epochs. Every number below checks plumbing only; none is a model result.")
if QUICK_RUN and FINAL_EVAL:   # stop now, not after training: section 11 would refuse anyway
    raise RuntimeError("FINAL_EVAL is refused while QUICK_RUN is on: a plumbing check must not open the held-out test split.")

The next cell picks the device, fixes the random seeds and names this run. A run has two folders, both ignored by Git:

- `artifacts/models/lineitems/<run_id>/`: the best checkpoint, the configuration and the training history;
- `artifacts/evaluation/<run_id>/`: every report, and every figure of this notebook under `figures/`.

One exception protects a finished run. If you reload a full run (`RELOAD_RUN_DIR`) with `QUICK_RUN = True`, the reports and figures of that execution go to `artifacts/evaluation/<run_id>-quick-check/`. Scores of a dozen pages can then never replace the full run's reports.

The `show(fig, name)` helper saves a figure there and then displays it, so the pictures you see are also on disk. `shown(path)` shortens a path for printing when it lies inside the repository.

> **Tip for long runs on a Mac.** macOS may put an idle machine to sleep in the middle of training. Start Jupyter with `caffeinate -i -s jupyter lab`, or keep the lid open and the machine awake. Training writes a checkpoint after every epoch either way.

In [ ]:
device = select_device(DEVICE or "auto")
set_seed(SEED)
reloaded_record = None
if RELOAD_RUN_DIR is not None:
    RUN_DIR = Path(RELOAD_RUN_DIR).expanduser().resolve()
    RUN_ID = RUN_DIR.name
    reloaded_record = json.loads((RUN_DIR / "manifest.json").read_text())
else:
    RUN_ID = new_run_id(SEED) + ("-quick" if QUICK_RUN else "")
    RUN_DIR = ARTIFACTS_ROOT / "models" / "lineitems" / RUN_ID
# "<run_id>", or "<run_id>-quick-check" when a quick execution reloads a full run (its reports stay untouched).
EVALUATION_DIR = ARTIFACTS_ROOT / "evaluation" / evaluation_folder_name(RUN_ID, quick_run=QUICK_RUN, reloaded=reloaded_record)
EXPORT_VERSION = f"lineitems/{EXPORT_NAME or RUN_ID + '-serving'}"     # the registry entry an export would write
FIGURES_DIR = EVALUATION_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
style_axes()

def show(fig, name):
    # Save the figure under artifacts/evaluation/<run_id>/figures/, then display it.
    fig.savefig(FIGURES_DIR / f"{name}.png", bbox_inches="tight", facecolor="white")
    plt.show()

def share(count, total):
    # "12 of 40 (30.0%)": a count is always shown with its denominator.
    return f"{count:,} of {total:,} ({count / total:.1%})" if total else f"{count} of 0"

def shown(path):
    # A path for printing: relative to the repository when it lies inside it, unchanged when it does not.
    path = Path(path)
    return path.relative_to(ROOT) if path.is_relative_to(ROOT) else path

print("Device:", device, "| seed:", SEED)
print("Run id:", RUN_ID)
print("Run folder       :", RUN_DIR)
print("Evaluation folder:", EVALUATION_DIR)
if EVALUATION_DIR.name != RUN_ID:
    print("  (a quick execution of a full run: its reports go to this separate folder, the full run's reports are not touched)")
show(pipeline_fig, "00_pipeline")

## 3 · Inputs

### 3.1 The M4 cache

Running OCR takes several seconds per page and needs the OCR container, so the 500 pages were read **once** by the real M4 adapter (`run_page_reading` with the served PaddleOCR models) and the results were stored. This is the *M4 cache*. Per page it holds:

- `document_page.json`: the validated `DocumentPage` record, exactly what M4 would send to M5;
- `rectified.png`: M4's *corrected page* image, the one M5 and M6 look at;
- `page_reading.json`: M4's own artifact, kept for audit.

`load_m4_cache(..., verify="hashes")` checks every file against its recorded SHA-256, and checks that the dataset images are the ones the cache was built from. If the cache is missing, the cell stops and prints the exact command that built it.

When `RELOAD_RUN_DIR` is set, the cell also compares the saved run with the present inputs. A run's scores belong to the labels and the M4 output it was trained on. If the label policy in the configuration cell, or the M4 cache, is not the one recorded in the run, the cell stops and says which differs. Scoring the run anyway would compare it with other truth.

In [ ]:
try:
    cache = load_m4_cache(CACHE_DIR, DATASET_DIR, verify="hashes", splits=("train", "validation"))
except M4CacheError as error:
    receipt = json.loads(CACHE_RECEIPT.read_text()) if CACHE_RECEIPT.is_file() else {}
    raise SystemExit(
        f"The M4 cache is missing or does not verify: {error}\n\n"
        "It is built by running the real M4 module over the dataset inside the OCR container.\n"
        f"The command recorded in {shown(CACHE_RECEIPT)} is:\n\n{receipt.get('build_command', '(receipt not found)')}\n\n"
        "After it finishes, check it with:\n"
        "  python -m pipelines.documents.m4_cache build --dataset <dataset> --cache <cache> --verify-only") from error

m4 = cache.manifest["m4"]
print("Cache        :", shown(cache.root), "| schema", cache.manifest["schema"])
print("Dataset      :", cache.manifest["dataset"]["name"], "|", cache.manifest["dataset"]["page_count"], "pages")
print("OCR weights  :", m4["versions"]["ocr_weights"], "| M4 code", m4["versions"]["m4_code"],
      "| page config", m4["versions"]["page_config"])
print("Built in     :", m4["container_image"], "from commit", str(m4["git_commit"])[:7], "(dirty tree)" if m4["git_dirty"] else "")
pages_table = pd.DataFrame([{ "split": e["split"], "page_type": e["page_type"], "M4 page status": e["page_status"]}
                            for e in cache.manifest["pages"]])
display(pages_table.groupby(["split", "page_type"])["M4 page status"].value_counts().unstack(fill_value=0))

if reloaded_record is not None:      # a saved run is scored only against the inputs it was trained on
    mismatches = reload_mismatches(reloaded_record, POLICY, cache)
    if mismatches:
        raise RuntimeError(f"The saved run {RUN_ID} cannot be evaluated with the present inputs:\n  - " + "\n  - ".join(mismatches)
                           + "\nRestore POLICY and CACHE_DIR to the run's values, or train a new run.")
    print("Reloaded run   : trained with this label policy and this M4 cache (manifest "
          + reloaded_record["context"]["m4_cache"]["manifest_sha256"][:12] + "…)")

**How to read the table.** The `test` rows are metadata only: page counts and M4's status per page. No text box, label or image of a test page is looked at before the opt-in final evaluation. Each claim has two pages: a *survey assessment* (no parts table) and a *repair estimate* (the parts table M5 cares about). `complete` means M4 read the page. `unreadable` means M4 **refused** the page: here PaddleOCR returned one text region nested inside another, and M4 fails such a page instead of guessing. An unreadable page is a failed page with zero text boxes. It is not an empty estimate. M5 has nothing to read on it, so every count below keeps these pages in the denominator as failures.

### 3.2 The dataset annotations

`annotations.jsonl` gives, per page, the ground-truth **field regions** (`regions`: label, true text, pixel box) and, separately, everything the synthetic surveyor added in ink (`marks`, `handwritten_words`). `claims.jsonl` gives the structured truth per claim (parts, quantities, amounts). The split column (train / validation / test) is inherited from the dataset and **grouped by claim**: both pages of a claim are always in the same split. The cell asserts that, because a claim split across training and validation would leak its vehicle and amounts.

In [ ]:
annotations = read_annotations(DATASET_DIR, splits=("train", "validation"))
claims = read_claims(DATASET_DIR, claim_ids={p.dataset_claim_id for p in cache.pages})
claims_per_split = assert_claims_in_one_split(cache.pages)   # raises if a claim crosses splits
print("Claims per split (no claim crosses a split):", claims_per_split)

# Of the test split only metadata is read before the opt-in final evaluation: its page count here and, in the
# table above, M4's status per page. No test page's boxes, labels or image are opened (test_pages does not exist yet).
train_pages_all, validation_pages_all = list(cache.split("train")), list(cache.split("validation"))
print("Pages:", {name: sum(e["split"] == name for e in cache.manifest["pages"]) for name in ("train", "validation", "test")})
unreadable = [(p.image_id, p.split, list(p.entry["reasons"])) for p in train_pages_all + validation_pages_all
              if p.page_status == "unreadable"]
print("Pages M4 could not read (train and validation):")
for row in unreadable:
    print("  ", *row)

example = cache.by_id(EXAMPLE_PAGE)
assert example.split == "train", "pictures before training must use a training page"
example_annotation = annotations[EXAMPLE_PAGE]
print("\nOne ground-truth region:", example_annotation["regions"][9])
print("One surveyor mark      :", {k: example_annotation["marks"][2][k] for k in ("class", "style", "bbox", "text", "revised_amount")})

### 3.3 The `DocumentPage` record

`DocumentPage` is the contract between M4 and M5 (`src/claim_cmev/contracts/documents.py`). The parts M5 uses:

| Field | Meaning |
| --- | --- |
| `text_boxes` | every located piece of text: `text`, `confidence` (OCR's own score), `box_norm`, `order_index` (reading order), `granularity` |
| `text_granularity` | `line` here: PaddleOCR returns **text lines**, not single words |
| `transform` | how the uploaded image maps to the corrected page (size, rotation, a 3×3 homography) |
| `quality` | `complete`, `partial` or `unreadable`, with reasons |
| `versions` | which OCR weights, configuration and code produced the record |
| `corrected_render_ref` | hash of the corrected page image, so nobody can swap the picture under the boxes |

**Concept: text box.** One rectangle of text that OCR found, with the string it read. On this layout a text box is usually one table cell or one form value.

**Concept: OCR confidence.** PaddleOCR's own estimate (0 to 1) that it read the characters correctly. It is not a calibrated probability. Section 4.4 shows a case where it stays high while the text is wrong.

**Concept: homography.** A 3 × 3 matrix that maps every point of one flat view of a page to another view of it. One matrix can express a shift, a rotation, a change of scale and the perspective tilt of a sheet photographed at an angle. M4 stores the homography that takes the uploaded image to the corrected page, so any box can be carried from one frame to the other.

In [ ]:
record = example.page
print("page_id:", record.page_id, "| text_granularity:", record.text_granularity, "| quality:", record.quality.state,
      "| boxes:", len(record.text_boxes), "| mean OCR confidence:", round(record.mean_text_confidence, 3))
print("transform:", {key: getattr(record.transform, key) for key in
                     ("source_width", "source_height", "corrected_width", "corrected_height", "geometry_correction")})
boxes_table = pd.DataFrame([{"order_index": b.order_index, "text": b.text, "confidence": round(b.confidence, 3),
                             "box_norm": tuple(round(v, 3) for v in b.box_norm), "granularity": b.granularity}
                            for b in sorted(record.text_boxes, key=lambda b: b.order_index)])
display(boxes_table.iloc[14:36])

**What to look for.** Rows 16 to 21 are the column headers. From row 22 on you see table cells: `Replace`, `460.00`, `Nett`, then the description. Notice three things, each a lesson of this notebook:

- The quantity cell (`1`) of every row is **missing**: OCR did not detect the single digit.
- Row 28 reads `150`, but the page prints `415.00`: a blue strike line crosses that amount.
- `' Not accepted'` and `SCD 585.00` are **handwriting** (the second should read `SGD 585.00`). They are text boxes like any other, and M5 must learn that they are not printed fields.

### 3.4 Coordinate frames

A box passes through five frames on its way into the model. Mixing two of them up is the most common silent bug in document models, so follow one box through all five.

| # | Frame | Units | Used by |
| --- | --- | --- | --- |
| 1 | Dataset image | pixels of the JPEG | the ground-truth annotations |
| 2 | M4 corrected page | pixels of `rectified.png` | OCR boxes, M6 detections |
| 3 | `box_norm` | 0 to 1 of the corrected page | every shared record |
| 4 | LayoutLMv3 `bbox` | integers 0 to 999 | the model's 2-D position embeddings |
| 5 | Visual input | 224 × 224 pixels | the model's image patches |

On this dataset M4 applied **no geometric correction**: frames 1 and 2 are identical (the transform is the identity). Real photographs are tilted and warped, and then they differ. The code below therefore still maps annotation boxes through `source_to_corrected_boxes`, so the notebook stays correct on real pages.

In [ ]:
amount_region = next(r for r in example_annotation["regions"] if r["label"] == "part_3_amount")
dataset_box = np.array(amount_region["bbox"], dtype=float)                         # frame 1
corrected_box = source_to_corrected_boxes(record, [dataset_box])[0]                # frame 2 (truth, mapped)
ordered_boxes = sorted(record.text_boxes, key=lambda b: b.order_index)
size = np.array([record.transform.corrected_width, record.transform.corrected_height] * 2)
m4_px = np.array([b.box_norm for b in ordered_boxes]) * size                       # M4 boxes in frame 2
hit = int(np.argmax(overlap_of_smaller(m4_px, [corrected_box])[:, 0]))             # the M4 box over that field
m4_box = ordered_boxes[hit]
grid_box = layout_bbox(m4_box.box_norm)                                            # frame 4
frames = pd.DataFrame([
    ("1 dataset image (truth)", "pixels", tuple(dataset_box.round(1))),
    ("2 corrected page (truth, mapped)", "pixels", tuple(corrected_box.round(1))),
    ("2 corrected page (M4 box)", "pixels", tuple(m4_px[hit].round(1))),
    ("3 box_norm (M4 box)", "0..1", tuple(round(v, 4) for v in m4_box.box_norm)),
    ("4 LayoutLMv3 bbox", "integers 0..999", grid_box),
    ("5 visual input", "224 px", tuple(round(v * 224, 1) for v in m4_box.box_norm))], columns=["frame", "units", "x0, y0, x1, y1"])
print(f"M4 read {m4_box.text!r} (confidence {m4_box.confidence:.3f}); the page prints {amount_region['text']!r}")
display(frames)

source_image, corrected_image = Image.open(example.source_path).convert("RGB"), Image.open(example.rectified_path).convert("RGB")
fig, axes = plt.subplots(1, 4, figsize=(15.5, 4.4), gridspec_kw={"width_ratios": [1.5, 1.5, 0.85, 0.85], "wspace": 0.3})
pad = np.array([-90, -40, 90, 40])
draw_boxes(axes[0], source_image, [dataset_box], [PALETTE["vermillion"]], extent=dataset_box + pad, linewidth=2)
axes[0].set_title("1 · Dataset image: true field box", fontsize=10)
draw_boxes(axes[1], corrected_image, [corrected_box, m4_px[hit]], [PALETTE["vermillion"], PALETTE["blue"]],
           extent=corrected_box + pad, linewidth=2)
axes[1].set_title("2 · Corrected page: truth (red), M4 box (blue)", fontsize=10)
axes[2].add_patch(plt.Rectangle((0, 0), 1, 1, fill=False, edgecolor="black"))
x0, y0, x1, y1 = m4_box.box_norm
axes[2].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, color=PALETTE["blue"]))
axes[2].set(xlim=(-0.05, 1.05), ylim=(1.05, -0.05), aspect=1754 / 1240, xlabel="x as a share of the page width", ylabel="y as a share of the page height")
axes[2].set_title("3, 4 · Position on the page", fontsize=10)
axes[2].annotate(f"box_norm\n{tuple(round(v, 3) for v in m4_box.box_norm)}\n× 1000, truncated →\nbbox {grid_box}", (x0, y1),
                 xytext=(0.05, 0.62), fontsize=7.5, arrowprops={"arrowstyle": "->"})
small = corrected_image.resize((224, 224), Image.BICUBIC)
draw_boxes(axes[3], small, [np.array(m4_box.box_norm) * 224], [PALETTE["blue"]], linewidth=1.5)
axes[3].set_title("5 · Visual input, 224 × 224", fontsize=10)
fig.suptitle(f"One text box of {EXAMPLE_PAGE} followed through the five coordinate frames", y=1.0)
show(fig, "01_coordinate_frames")

**What to look for.** In panel 2 the blue M4 box is a little larger than the red ground-truth box: OCR boxes include some margin. That is why the alignment in section 4 compares boxes by *overlap* and not by equality. Panel 5 shows how little the visual branch can see: the whole A4 page squeezed into 224 × 224 pixels. At that size digits are a smudge. The image tells the model about layout and ink, and the **text comes from OCR**, not from the picture.

## 4 · Understanding the data

All pictures and statistics in this section use the **training** split (some tables add validation). Nothing here reads the test split.

### 4.1 One page as M4 delivers it

Left: every M4 text box, numbered in reading order and coloured by OCR confidence. Right: the same page with the dataset's ground truth: printed field values (coloured by field type) and the surveyor's handwritten words (black dashes).

In [ ]:
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from matplotlib.lines import Line2D

confidence_map, confidence_scale = plt.get_cmap("viridis_r"), Normalize(vmin=0.80, vmax=1.0)
fig, axes = plt.subplots(1, 2, figsize=(15, 10.5))
draw_boxes(axes[0], corrected_image, m4_px, [confidence_map(confidence_scale(b.confidence)) for b in ordered_boxes],
           texts=[str(b.order_index) for b in ordered_boxes], linewidth=1.6, text_size=6.5)
axes[0].set_title(f"{EXAMPLE_PAGE}: {len(ordered_boxes)} M4 text boxes in reading order, coloured by OCR confidence")
bar = fig.colorbar(ScalarMappable(norm=confidence_scale, cmap=confidence_map), cax=axes[0].inset_axes([0.12, 0.42, 0.5, 0.012]),
                   orientation="horizontal")
bar.set_label("OCR confidence (colour scale starts at 0.80)", fontsize=8)
truth_px = source_to_corrected_boxes(record, [r["bbox"] for r in example_annotation["regions"]])
truth_types = [field_type(r["label"]) for r in example_annotation["regions"]]
draw_boxes(axes[1], corrected_image, truth_px, [TYPE_COLOURS[t] for t in truth_types], linewidth=1.8)
for word in example_annotation["handwritten_words"]:
    x0, y0, x1, y1 = source_to_corrected_boxes(record, [word["bbox"]])[0]
    axes[1].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="black", linestyle="--", linewidth=1.2))
axes[1].legend(handles=[Line2D([], [], color=TYPE_COLOURS[t], lw=2, label=t) for t in TYPE_ORDER if t in set(truth_types)]
               + [Line2D([], [], color="black", lw=1.2, ls="--", label="handwritten word (surveyor ink)")],
               loc="center", bbox_to_anchor=(0.35, 0.47), title="Ground truth", frameon=True)
axes[1].set_title("Ground truth: printed field values and surveyor handwriting")
show(fig, "02_page_as_m4_delivers_it")

**What to look for.**

- **Left.** Almost every box is dark (high confidence). Reading order runs top to bottom, but inside a table row it is *not* left to right: OCR sorted the cells of row 1 as `Replace`, `460.00`, `Nett` and only then the description. A model that relied on token order alone would be confused, which is one reason LayoutLMv3 is given the box positions.
- **Right.** Each printed value has one ground-truth box. There is a purple quantity box in every row on the right, and **no** numbered box over it on the left.
- The handwritten words sit directly under the printed remark `Nett`. Where the handwriting touches the print, OCR sometimes returns **one** box for both. Section 4.2 deals with that.

### 4.2 From ground truth to training labels: the alignment

**The design decision of this notebook (proposed).** M4 returns text *lines*, so the unit this model classifies is **one M4 text box**. The model reads the whole text of the box, split into subword tokens that all carry the box's coordinates, and predicts **one label for the box**. No word positions are invented. `docs/document-neural-workflow.md` calls this the "region-level representation" and says it "needs its own measured justification"; the measurements in this notebook are that justification, on synthetic pages. It also matches how LayoutLMv3 was pretrained: its authors gave every word the box of its text *segment* (line), not a box of its own.

**Concept: BIO labels.** `B-x` marks the first box of field `x`, `I-x` a following box of the same field, and `O` ("outside") a box that belongs to no field. Classic BIO works on words; here it works on boxes.

To train, each M4 box needs a label. The dataset labels *field regions*, not OCR boxes, so the two are compared geometrically in the corrected-page frame. Each box gets one of three outcomes:

| Outcome | When | Training label |
| --- | --- | --- |
| **field** | the box significantly overlaps exactly one field region and no handwritten text | `B-<label>`, or `I-<label>` when an earlier box already started that field |
| **other** | it overlaps no field region: form labels, headers, handwriting, the footer | `O` |
| **masked** | ambiguous: it overlaps two or more field regions, or a printed field **and** handwritten text | `-100`: excluded from the loss and from the scores, and counted |

Two rules follow from the project invariants:

- **Surveyor ink is separate evidence.** A handwritten replacement amount is `O`. It is never `part_N_amount`, because the printed (declared) amount and the surveyor's revised amount must never be confused.
- **The label says which field a box is, not whether OCR read it correctly.** A printed amount struck through by the surveyor keeps its `part_N_amount` label even when OCR misread the digits. Reading quality is measured separately in 4.4.

**"Significantly overlaps" needs a number.** The measure is *intersection area divided by the area of the smaller of the two boxes*. It is 1 when one box lies inside the other, which is the normal case (an OCR line slightly larger than its field), and small when two neighbours merely touch. The threshold is chosen from the **training split only**. The next cell shows the distribution that motivated it.

In [ ]:
print(inspect.getsource(overlap_of_smaller))
samples = overlap_samples(train_pages_all, annotations)      # refuses anything but training pages
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
bins = np.linspace(0, 1, 21)
for ax, key, threshold, title, xlabel in (
        (axes[0], "field_overlap", POLICY.field_overlap_min, "M4 box against a field region",
         "intersection / area of the smaller box"),
        (axes[1], "handwriting_overlap", POLICY.handwriting_overlap_min, "M4 box against a handwritten word",
         "share of the handwritten word inside the M4 box")):
    values = samples[key]
    ax.hist(values, bins=bins, color=PALETTE["blue"])
    ax.axvline(threshold, color=PALETTE["vermillion"], linestyle="--", label=f"threshold {threshold:g} (proposed)")
    ax.set_yscale("log")
    ax.set(xlabel=xlabel, ylabel="touching pairs (log scale)",
           title=f"{title}\n{share(int((values >= threshold).sum()), len(values))} at or above the threshold")
    ax.legend(loc="upper center")
fig.suptitle("Training split only: how strongly M4 boxes overlap what they touch", y=1.04)
show(fig, "03_overlap_distributions")
for key, low, high in (("field_overlap", 0.3, 0.7), ("handwriting_overlap", 0.3, 0.7)):
    values = samples[key]
    print(f"{key}: {share(int(((values >= low) & (values < high)).sum()), len(values))} pairs lie between {low} and {high}")

**What to look for.** Both histograms have a log axis and both are **two-peaked**: nearly every pair either overlaps almost completely (right edge) or barely touches (left edge). The valley between is almost empty, so the exact threshold matters little: 0.5, in the middle of the valley, is the proposed value for both. The pairs on the left of the first plot are mostly handwriting written just under a printed remark, brushing its box.

The decision itself is a dozen lines. Read them: this is the whole labelling rule.

In [ ]:
source = inspect.getsource(align_page)
start, stop = source.index("    decided:"), source.index("    fields: list[FieldMatch]")
print(source[start:stop])
print(POLICY)

Now the rule is applied to every training and validation page. The overlay shows the outcome for one training page that contains a merged box; the tables count outcomes for all pages.

In [ ]:
alignments = align_pages(train_pages_all + validation_pages_all, annotations, POLICY)
box_table = pd.DataFrame(alignment_box_rows(alignments))
field_table = pd.DataFrame(field_rows(alignments))
OUTCOME_COLOURS = {"masked": TYPE_COLOURS["masked"], "other": TYPE_COLOURS["O"]}

def outcome_colour(box):
    return TYPE_COLOURS[field_type(box.field)] if box.outcome == "field" else OUTCOME_COLOURS[box.outcome]

def outcome_legend(ax, **kwargs):
    names = [t for t in TYPE_ORDER if t != "O"]
    ax.legend(handles=[Line2D([], [], color=TYPE_COLOURS[t], lw=2.5, label=f"field: {t}") for t in names]
              + [Line2D([], [], color=TYPE_COLOURS["O"], lw=2.5, label="O (no field)"),
                 Line2D([], [], color=TYPE_COLOURS["masked"], lw=2.5, label="masked (ambiguous)")], **kwargs)

merged_page_id = box_table[(box_table.split == "train") & (box_table.outcome == "masked")].image_id.iloc[0]
merged_page, merged_alignment = cache.by_id(merged_page_id), alignments[merged_page_id]
last_row = max(b.box_px[3] for b in merged_alignment.boxes if row_slot(b.field) is not None)
fig, ax = plt.subplots(figsize=(11, 11 * (last_row - 110) / 1160))
draw_boxes(ax, Image.open(merged_page.rectified_path).convert("RGB"), [b.box_px for b in merged_alignment.boxes],
           [outcome_colour(b) for b in merged_alignment.boxes], linewidth=1.8,
           texts=[(b.label or "masked") if b.outcome != "other" else None for b in merged_alignment.boxes], text_size=6,
           extent=(40, 150, 1200, last_row + 40))
outcome_legend(ax, loc="upper center", bbox_to_anchor=(0.5, -0.01), ncol=4, fontsize=8)
ax.set_title(f"Training labels of {merged_page_id} (training split): one outcome per M4 box")
show(fig, "04_alignment_overlay")
display(box_table.groupby(["split", "outcome", "reason"]).size().rename("M4 boxes").to_frame())

**What to look for.** Coloured boxes carry the field label written above them. Grey boxes are `O`: form labels, column headers and handwriting that stands alone. A yellow box is **masked**: there OCR merged a printed remark with the handwriting under it, and no honest single label exists. Masked boxes are rare, and the table counts them so they cannot disappear.

**Missing evidence is reported, not hidden.** The other direction matters just as much: for each ground-truth field, did M4 deliver a box at all?

- `matched`: at least one box carries the field's label;
- `ambiguous`: the field's only box was masked;
- `missed`: no M4 box overlaps the field, so OCR did not detect it;
- `page_unreadable`: M4 refused the whole page.

In [ ]:
row_fields = field_table[field_table.field_type != "page_field"]
delivery = (row_fields.groupby(["split", "field_type", "status"]).size().unstack(fill_value=0)
            .reindex(columns=["matched", "ambiguous", "missed", "page_unreadable"], fill_value=0))
delivery["fields"] = delivery.sum(axis=1)
delivery["matched share"] = (delivery["matched"] / delivery["fields"]).round(3)
display(delivery)

fig, ax = plt.subplots(figsize=(9, 3.6))
train_delivery = delivery.loc["train"].reindex(["quantity", "description", "action", "amount", "remarks"])
left = np.zeros(len(train_delivery))
for status, colour in (("matched", PALETTE["blue"]), ("ambiguous", PALETTE["yellow"]), ("missed", PALETTE["vermillion"]),
                       ("page_unreadable", PALETTE["grey"])):
    values = (train_delivery[status] / train_delivery["fields"]).to_numpy()
    ax.barh(train_delivery.index, values, left=left, color=colour, label=status)
    left += values
for i, (name, row) in enumerate(train_delivery.iterrows()):
    ax.text(1.01, i, f"{int(row['matched'])} of {int(row['fields'])} matched", va="center", fontsize=8)
ax.set(xlim=(0, 1.32), xlabel="share of ground-truth table cells (training split)",
       title="What M4 delivered for each kind of table cell (training split)")
ax.invert_yaxis(); ax.legend(loc="upper center", bbox_to_anchor=(0.4, -0.2), ncol=4, fontsize=8)
show(fig, "05_field_delivery")

per_label = field_table.groupby(["label", "split", "status"]).size().unstack(["split", "status"], fill_value=0).sort_index(axis=1)
per_label.to_csv(EVALUATION_DIR / "alignment_per_label.csv")
print(f"Per-label counts by split for all {len(per_label)} field labels are saved to alignment_per_label.csv.")
print("Excerpt: the table cells of rows 1 and 8.")
display(per_label[per_label.index.str.match(r"part_[18]_")])

**A finding about the M4 → M5 hand-off.** The **quantity** bar is mostly red: OCR does not detect the small single digit in the `Qty` column on most rows. No M5 model can label a box that does not exist, so on these pages most quantities cannot be recovered from this M4 output. The project rule is that a missing quantity stays **null**: it is never assumed to be one. This is a question for M4 (a text-detection threshold), not something M5 should patch, and it is reported as such at the end of this notebook.

Remarks are the only cells with `ambiguous` entries: those are the print-plus-handwriting merges.

**A gallery of the interesting cases.** Each panel is a crop of a training page. The box named in the title is drawn thick.

In [ ]:
quality = pd.DataFrame(text_quality_rows(alignments, annotations))
train_boxes = box_table[box_table.split == "train"]
struck = quality[(quality.split == "train") & (quality.field_type == "amount") & quality.ink_over_field & ~quality.exact].iloc[0]
struck_alignment = alignments[struck.image_id]
struck_box_id = struck_alignment.field(struck.label).box_ids[0]
cases = [("Print merged with handwriting → masked", train_boxes[train_boxes.reason == "print_and_handwriting"].iloc[0]),
         ("Struck printed amount → still its field", train_boxes[(train_boxes.image_id == struck.image_id) & (train_boxes.box_id == struck_box_id)].iloc[0]),
         ("Handwritten remark → O", train_boxes[train_boxes.reason == "handwriting"].iloc[1]),
         ("One field in two boxes → B- then I-", train_boxes[train_boxes.label.fillna("").str.startswith("I-")].iloc[0])]
fig, axes = plt.subplots(2, 2, figsize=(14, 5.0))
for ax, (title, case) in zip(axes.flat, cases):
    alignment = alignments[case.image_id]
    image = Image.open(cache.by_id(case.image_id).rectified_path).convert("RGB")
    x0, y0, x1, y1 = case.box_px
    centre = (x0 + x1) / 2
    extent = (max(0, centre - 360), max(0, y0 - 70), min(image.width, centre + 360), min(image.height, y1 + 70))
    draw_boxes(ax, image, [b.box_px for b in alignment.boxes], [outcome_colour(b) for b in alignment.boxes],
               linewidth=1.2, extent=extent)
    ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, linewidth=3.2,
                               edgecolor=outcome_colour(next(b for b in alignment.boxes if b.box_id == case.box_id))))
    ax.set_title(f"{title}\n{case.image_id}: OCR read {case.text!r} → {case.label or 'masked'}", fontsize=9)
outcome_legend(fig, loc="lower center", ncol=8, fontsize=8, bbox_to_anchor=(0.5, -0.06))
fig.suptitle("Alignment cases on training pages (thick box = the case)", y=1.02)
show(fig, "06_alignment_cases")
print(f"Struck amount: page prints {struck.truth!r}, OCR read {struck.read!r} with confidence {struck.confidence:.3f}; label {struck.label}")

### 4.3 Which label names exist: the vocabulary

The label list is `O` plus `B-`/`I-` for every field label of the dataset, sorted. It is built from the **names** used in the training and validation annotations. Names are schema, not page content, so reading them is not data leakage; the test pages themselves stay closed.

**Concept: row-slot labels.** The table cells are named `part_1_description`, `part_1_amount`, `part_2_description`, …: the label carries the **row number on the page** (the *slot*). This is convenient, because the serving adapter can group boxes into rows just by reading the label, with no geometry. It has two weaknesses you should keep in mind:

1. A model can never output a row beyond the largest `N` it was trained with (here 8), and rows with a high `N` have few examples.
2. The label ties meaning to position: "third row" is learned from where the box sits on *this* layout. On another layout, or a table that starts lower on the page, the slot could be wrong while the field type is right.

The alternative is to predict only the field type (`description`, `amount`, …) and to group boxes into rows by geometry afterwards. That is future work, named in section 13; this notebook keeps slot labels so that the existing adapter can read the result.

In [ ]:
label_names = region_label_names(annotations)            # names from train + validation annotations only
labels = bio_labels(label_names)
label2id = {label: index for index, label in enumerate(labels)}
print(f"{len(label_names)} field names → {len(labels)} labels: O plus B-/I- for each.  First five: {labels[:5]}")

labelled = box_table[box_table.label.notna()]
support = labelled.groupby(["split", "label"]).size().unstack(0, fill_value=0).reindex(labels, fill_value=0)
ordered = support.sort_values("train", ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2.3, 1]})
colours = [TYPE_COLOURS[field_type(field_name(name))] for name in ordered.index]
axes[0].bar(range(len(ordered)), ordered["train"].clip(lower=0.5), color=colours, width=0.9)
axes[0].set_yscale("log")
axes[0].set(xlabel=f"the {len(labels)} labels, most frequent first (bars at 0.5 = no training box at all)",
            ylabel="training boxes (log scale)", xticks=[],
            title=f"Training boxes per label: {int((support['train'] == 0).sum())} of {len(labels)} labels have none")
axes[0].legend(handles=[Line2D([], [], color=TYPE_COLOURS[t], lw=6, label=t) for t in TYPE_ORDER], ncol=4, fontsize=8)
rows_per_page = (field_table[field_table.field_type == "amount"].groupby(["split", "slot"]).image_id.nunique().unstack(0, fill_value=0))
rows_per_page.index = rows_per_page.index.astype(int)
rows_per_page[["train", "validation"]].plot.bar(ax=axes[1], color=[PALETTE["blue"], PALETTE["orange"]], width=0.8)
axes[1].set(xlabel="row slot N of part_N_…", ylabel="estimate pages that have this row", title="Few pages have rows 7 and 8")
axes[1].tick_params(axis="x", rotation=0)
show(fig, "07_label_support")
print("Labels with no training box:", int((support["train"] == 0).sum()), "| of them I- labels:",
      int(((support["train"] == 0) & support.index.str.startswith("I-")).sum()))
display(support[support.index.str.startswith("I-") & (support.sum(axis=1) > 0)])

**What to look for.** The left chart has a log axis. `O` is by far the largest class (the first bar): about half of all boxes are static text. That is why section 8 reports `O` separately and never lets it into the headline score. The many bars at the right edge are labels with **no training example**: almost all `I-` labels, because a field is nearly always one box. The model cannot learn them; the few `I-` examples that exist are listed under the chart. On the right, every estimate has rows 1 to 3, and fewer and fewer pages have rows 4 to 8, so the high slots are learned from fewer examples.

### 4.4 How well did OCR *read* the boxes?

Labelling a box and reading it are different problems, done by different modules. M5 decides *which box is the amount*. Whether the digits in that box are right was decided earlier, by M4. Compare OCR text with the true text for every matched field:

- **exact match**: the two strings are equal (after collapsing whitespace);
- **CER (character error rate)**: edit distance divided by the length of the true text. 0 is perfect; 0.5 means half the characters had to be changed.

Amounts are then split into *clean* and *ink over the amount*. That is decided by geometry: some stroke or handwritten part of a surveyor mark crosses the amount's box. The dataset's own list `source_amount_occlusions` names only the price-change marks; a row strike also runs straight through the amount, so geometry is the more complete test.

In [ ]:
by_type = (quality.groupby(["split", "field_type"]).agg(fields=("exact", "size"), exact=("exact", "sum"), mean_cer=("cer", "mean")))
by_type["exact share"] = (by_type["exact"] / by_type["fields"]).round(3)
by_type["mean_cer"] = by_type["mean_cer"].round(4)
display(by_type)
quantities = quality[(quality.field_type == "quantity") & (quality.split == "train")]
misread_quantities = quantities[~quantities.exact]
print(f"Quantity cells M4 delivered on training pages: {len(quantities)} | read exactly: {share(int(quantities.exact.sum()), len(quantities))}")
print("The printed quantity of the misread ones, and what OCR returned:",
      {f"{truth!r} read as {read!r}": count for (truth, read), count in Counter(zip(misread_quantities.truth, misread_quantities.read)).most_common()},
      f"| their OCR confidence: {misread_quantities.confidence.min():.2f} to {misread_quantities.confidence.max():.2f}")

amounts = quality[(quality.field_type == "amount") & (quality.split == "train")].copy()
amounts["ink"] = np.where(amounts.ink_over_field, "ink over the amount", "clean")
ink_table = amounts.groupby("ink").agg(amounts=("exact", "size"), exact=("exact", "sum"), mean_confidence=("confidence", "mean"))
ink_table["exact share"] = (ink_table["exact"] / ink_table["amounts"]).round(3)
display(ink_table)
style_table = amounts.groupby("row_style").agg(amounts=("exact", "size"), exact=("exact", "sum"), mean_confidence=("confidence", "mean"))
style_table["exact share"] = style_table["exact"] / style_table["amounts"]
style_table = style_table.sort_values("exact share")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].barh(style_table.index, style_table["exact share"], color=PALETTE["blue"])
for i, (name, row) in enumerate(style_table.iterrows()):
    axes[0].text(0.02, i, f"{int(row['exact'])} of {int(row['amounts'])}", va="center", fontsize=8, color="white")
axes[0].set(xlim=(0, 1), xlabel="share of printed amounts OCR read exactly", ylabel="surveyor mark on the row",
            title="Printed amounts read exactly, by the mark on the row (training)")
wrong, right = amounts[~amounts.exact].confidence, amounts[amounts.exact].confidence
bins = np.linspace(0.5, 1.0, 26)
axes[1].hist(right, bins=bins, color=PALETTE["blue"], alpha=0.75, label=f"read exactly ({len(right)})")
axes[1].hist(wrong, bins=bins, color=PALETTE["vermillion"], alpha=0.85, label=f"misread ({len(wrong)})")
axes[1].set_yscale("log")
axes[1].set(xlabel="OCR confidence of the amount box", ylabel="printed amounts (log scale)",
            title=f"OCR confidence does not separate misreads: {share(int((wrong >= 0.9).sum()), len(wrong))}\nmisread amounts have confidence 0.9 or more")
axes[1].legend(loc="upper left")
show(fig, "08_ocr_text_quality")
display(amounts[~amounts.exact & (amounts.confidence >= 0.95)][["image_id", "label", "truth", "read", "confidence", "row_style"]].head(6))

**What to look for.**

- **Quantity is the worst-read field type.** In the first table, compare the `exact share` and the `mean_cer` of `quantity` with the other rows of the same split. M4 delivers few quantity cells (section 4.2), and of those it delivers it misreads about one in eight. The printout under the table says as what: the printed `1` mostly comes back as `7`, and sometimes as the letter `I`. The two mistakes have very different consequences. `I` is not a number, so the parser will refuse it and the quantity stays null. `7` **is** a number, and nothing later in the pipeline can tell that it is wrong by looking at it. Section 10.1 counts how many of these reach a `LineItem` record.
- Clean amounts are read almost perfectly. Amounts with **ink over them** are read much worse, and how much worse depends on the kind of mark: a cross drawn over the digits (`amount_cross`) is the hardest.
- The right-hand plot is the lesson to remember: many **misread** amounts still have a **high OCR confidence**. The table below it lists some. A strike line turns `415.00` into `150`, and the OCR engine is quite sure about `150`. So a confidence threshold cannot be the only guard on a printed amount.

This is why the system keeps things separate: M5 points at the amount box and reports the text M4 read; the exact decimal parser rejects strings that are not clean numbers; M6 detects the pen mark on the row; and a human confirms. No stage silently "repairs" a number. Keep the limit of these guards in mind: a parser checks the *form* of a value. A misread that is itself a well-formed number (`150` for `415.00`, `7` for `1`) passes every format check. Only a comparison with something else can catch it: the totals printed on the page, a pen mark on the row, or a person.

### 4.5 From boxes to tokens

LayoutLMv3 does not read boxes or words. It reads **tokens**: pieces of words from a fixed vocabulary of about 50,000 (byte-pair encoding, the same family as RoBERTa). The *processor* does three jobs:

1. splits each box's text into subword tokens and adds a start token `<s>` and an end token `</s>`;
2. gives every token the `bbox` of the box it came from (0–999 integers);
3. resizes the page image to 224 × 224 and normalises it.

**Concept: the first-subword rule.** A box of several tokens still gets only **one** label. The label is placed on the box's **first** token; all other tokens get `-100`, which the loss ignores. At prediction time the model's answer for a box is read at that same first token.

The serving adapter and this notebook call the *same* function for this, `encode_page`. Here is all of it:

In [ ]:
print(inspect.getsource(encode_page))
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained(BASE_CHECKPOINT, revision=BASE_REVISION, apply_ocr=False, local_files_only=OFFLINE)

example_alignment = alignments[EXAMPLE_PAGE]
words = page_words(record)                                   # boxes in reading order, text with whitespace collapsed
encoding = encode_page(processor, corrected_image, words, example_alignment.label_ids(label2id))
real_tokens = int(encoding["attention_mask"][0].sum())
token_table = pd.DataFrame({
    "token": processor.tokenizer.convert_ids_to_tokens(encoding["input_ids"][0][:real_tokens]),
    "box index": pd.array([None if w is None else words.readable[w] for w in encoding.word_ids(0)[:real_tokens]], dtype="Int64"),
    "bbox (0..999)": [tuple(b) for b in encoding["bbox"][0][:real_tokens].tolist()],
    "label id": encoding["labels"][0][:real_tokens].tolist()})
token_table["label"] = [labels[i] if i >= 0 else "· ignored (-100)" for i in token_table["label id"]]
print(f"{EXAMPLE_PAGE}: {len(words.texts)} boxes → {real_tokens} tokens (of {encoding['input_ids'].shape[1]} positions; the rest is padding).",
      f"{int((encoding['labels'][0] != IGNORE_INDEX).sum())} tokens carry a label.")
first_cell = next(i for i, b in enumerate(example_alignment.boxes) if row_slot(b.field) is not None)   # first table cell
start = int(token_table.index[token_table["box index"] == first_cell][0])
display(pd.concat([token_table.iloc[:4], token_table.iloc[start:start + 22]]))

**How to read the token table.** `Ġ` stands for a space in front of the token. The tokenizer puts a space before the text of every box, so the **first token of every box** starts with `Ġ`, and so does each further word inside a box. A token without `Ġ` continues the word before it (`ĠN` + `ett` is `Nett`). Tokens with the same *box index* come from one M4 box and share its `bbox`. Only the first token of each box has a label; the others show `-100`. Special tokens (`<s>`, `</s>`) have no box and no label.

**Sequence length.** The model reads at most **512 tokens** at once. A page with more is cut into overlapping *windows* (see the picture after this cell). The histogram shows how long the pages of this dataset actually are. Building the datasets here encodes every page once and keeps the tensors in memory.

In [ ]:
def quick_subset(pages, count):
    # First `count` pages (whole claims), plus the claims of any page M4 could not read, so that path is exercised too.
    keep = {p.dataset_claim_id for p in pages[:count]} | {p.dataset_claim_id for p in pages if p.page_status == "unreadable"}
    return [p for p in pages if p.dataset_claim_id in keep]

train_pages = quick_subset(train_pages_all, QUICK_PAGES["train"]) if QUICK_RUN else train_pages_all
validation_pages = quick_subset(validation_pages_all, QUICK_PAGES["validation"]) if QUICK_RUN else validation_pages_all
started = time.perf_counter()
train_set = PageWindowDataset(train_pages, processor, alignments=alignments, label2id=label2id)
validation_set = PageWindowDataset(validation_pages, processor, alignments=alignments, label2id=label2id)
print(f"Encoded {len(train_pages)} training and {len(validation_pages)} validation pages in {time.perf_counter() - started:.1f} s")
print(f"Training windows: {len(train_set)} from {len(train_set.pages)} pages | skipped (no text): {train_set.skipped}")
print(f"Validation windows: {len(validation_set)} from {len(validation_set.pages)} pages | skipped (no text): {validation_set.skipped}")
if QUICK_RUN:
    print("QUICK RUN subset: not a model result.")

token_counts = np.array(train_set.token_counts() + validation_set.token_counts())
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(token_counts, bins=30, color=PALETTE["blue"])
ax.axvline(512, color=PALETTE["vermillion"], linestyle="--", label="model limit: 512 tokens")
ax.set(xlim=(0, 540), xlabel="tokens per page (including <s> and </s>)", ylabel="pages",
       title=f"Page length in tokens: {token_counts.min()} to {token_counts.max()}, median {int(np.median(token_counts))}; "
             f"{int((token_counts >= 512).sum())} pages reach the limit")
ax.legend()
show(fig, "09_sequence_lengths")

**What to look for.** Every page of this dataset fits into **one** window with room to spare, so no window logic is exercised by the real data. A real multi-page or dense estimate can exceed 512 tokens, and the serving adapter handles that, so you should know how it works.

**Concept: overflow windows and stride.** A long page is cut into windows of 512 tokens. Each window repeats the last `stride = 64` tokens of the previous one, so that a box cut at a window border is seen whole in one of them. A box is labelled once: in the first window that contains its first token. To *see* this, the next cell builds an artificially long page by repeating the example page's boxes six times. It is an illustration, not data.

In [ ]:
repeat = 6
long_words = PageWords(words.boxes * repeat, tuple(range(len(words.texts) * repeat)), words.texts * repeat, words.bboxes * repeat)
long_encoding = encode_page(processor, corrected_image, long_words, [0] * len(long_words.texts))
window_count = long_encoding["input_ids"].shape[0]
fig, ax = plt.subplots(figsize=(11, 1.2 + 0.5 * window_count))
first_seen = set()
for window in range(window_count):
    ids = [w for w in long_encoding.word_ids(window) if w is not None]
    labelled_here = {ids_w for ids_w, lab in zip(long_encoding.word_ids(window), long_encoding["labels"][window].tolist())
                     if ids_w is not None and lab != IGNORE_INDEX}
    new = labelled_here - first_seen
    ax.barh(window, max(ids) - min(ids) + 1, left=min(ids), color=PALETTE["sky"], edgecolor="black",
            label="boxes inside the window" if window == 0 else None)
    ax.barh(window, max(new) - min(new) + 1, left=min(new), height=0.35, color=PALETTE["blue"],
            label="boxes whose label is read from this window" if window == 0 else None)
    first_seen |= labelled_here
    ax.text(max(ids) + 2, window, f"{int(long_encoding['attention_mask'][window].sum())} tokens", va="center", fontsize=8)
ax.set(yticks=range(window_count), yticklabels=[f"window {i}" for i in range(window_count)], xlabel="box index in reading order",
       title=f"Illustration: {len(long_words.texts)} boxes need {window_count} overlapping windows of at most 512 tokens (stride 64)")
ax.set_xlim(0, len(long_words.texts) * 1.14); ax.invert_yaxis(); ax.legend(loc="upper right", fontsize=8)
show(fig, "10_overflow_windows")
print("pixel_values for these windows:", tuple(long_encoding["pixel_values"].shape), "→ the same page image is repeated for every window")
print("Every box is labelled in some window:", first_seen == set(range(len(long_words.texts))))

**What to look for.** The light bars overlap: the end of one window is repeated at the start of the next. The dark bars do not overlap and together cover every box exactly once: that is the "first window wins" rule.

> **A version trap, fixed in `encode_page`.** When a window starts in the middle of a box's text, transformers 5 gives that leftover subword the box's label, while transformers 4 gave it `-100`. Serving never reads that position, so training must not learn from it. `encode_page` resets those positions to `-100` (its last line), and a unit test checks it.

## 5 · The model: LayoutLMv3 in plain terms

A *transformer* turns a sequence of input vectors into a sequence of output vectors, letting every position look at every other position ("attention"). LayoutLMv3 feeds it **three kinds of information at once**:

1. **Text.** Each subword token becomes a vector (the *text embedding*), plus a vector for its position in the sequence.
2. **Layout.** Each token also gets vectors for the x and y coordinates, the width and the height of its box (the *2-D position embedding*). This is what lets the model learn "the amount is the number in the fifth column".
3. **Image.** The 224 × 224 page image is cut into 14 × 14 = 196 patches of 16 × 16 pixels. Each patch becomes a vector too and is appended to the sequence, so text tokens can look at the picture around them (lines, ink, bold headers).

Twelve transformer layers mix all of this. On top sits a small **classification head**: for every text position it outputs one score per label (a *logit*). The patches are used inside the transformer but get no label.

```mermaid
flowchart LR
    T["box text → subword tokens<br/>input_ids [B, 512]"] --> E["text + 2-D position embeddings"]
    B["box coordinates 0..999<br/>bbox [B, 512, 4]"] --> E
    I["page image 224×224<br/>pixel_values [B, 3, 224, 224]"] --> P["196 patch embeddings"]
    E --> X["transformer encoder<br/>12 layers, 768 wide"]
    P --> X
    X --> H["classification head<br/>logits [B, 512, K]"]
    H --> L["label of each box =<br/>argmax at its first token"]
```

**What is pretrained and what is new.** Microsoft pretrained the embeddings and the encoder on 11 million scanned documents, to predict masked words and masked image patches. That knowledge (what text on forms looks like) is reused. Only the classification head is new: it starts with random numbers and must learn our `K` labels from 350 pages. Fine-tuning updates **all** weights, with a small learning rate so the pretrained knowledge is adjusted, not destroyed.

**Licence.** The model card of `microsoft/layoutlmv3-base` states `cc-by-nc-sa-4.0`: **non-commercial** use, and derivatives (these fine-tuned weights) carry the same terms. The export writes this into `NOTICE.md`.

In [ ]:
config = TrainConfig(seed=SEED, device=str(device), offline=OFFLINE, **RECIPE)
if QUICK_RUN:
    config.epochs = QUICK_EPOCHS
model, processor, loading = build_model(labels, config)
print("Tensors the pretrained checkpoint did NOT contain (they start random):", sorted(loading["missing_keys"]))
print("Tensors of the checkpoint that were not used:", sorted(loading["unexpected_keys"]) or "none")
blocks = pd.DataFrame(parameter_blocks(model))
blocks["share"] = (blocks["parameters"] / blocks["parameters"].sum()).map("{:.2%}".format)
display(blocks)
print(f"Total parameters: {blocks['parameters'].sum():,} | labels K = {len(labels)} | base revision {BASE_REVISION[:12]} | licence {BASE_LICENCE}")

fig, axes = plt.subplots(2, 1, figsize=(12, 6.6), gridspec_kw={"height_ratios": [1.25, 1], "hspace": 0.35})
ax = axes[0]
parts = [(0.0, 2.0, "box text → tokens\ninput_ids [B, 512]", "#56B4E9"), (0.0, 1.0, "box coordinates 0..999\nbbox [B, 512, 4]", "#56B4E9"),
         (0.0, 0.0, "page image 224×224\npixel_values [B, 3, 224, 224]", "#56B4E9"),
         (3.0, 1.5, "text + 2-D position\nembeddings", "#DDDDDD"), (3.0, 0.0, "196 image patch\nembeddings", "#DDDDDD"),
         (6.0, 0.75, "transformer encoder\n12 layers (pretrained)", "#DDDDDD"),
         (9.0, 0.75, f"classification head (new)\nlogits [B, 512, {len(labels)}]", "#E69F00"),
         (12.0, 0.75, "one label per box:\nargmax at its first token", "#F0E442")]
for x, y, text, colour in parts:
    ax.add_patch(FancyBboxPatch((x, y), 2.5, 0.8, boxstyle="round,pad=0.03", facecolor=colour, edgecolor="black"))
    ax.text(x + 1.25, y + 0.4, text, ha="center", va="center", fontsize=8.5)
for (x0, y0), (x1, y1) in [((2.55, 2.4), (3.0, 2.1)), ((2.55, 1.4), (3.0, 1.8)), ((2.55, 0.4), (3.0, 0.4)), ((5.55, 1.9), (6.0, 1.3)),
                           ((5.55, 0.4), (6.0, 1.0)), ((8.55, 1.15), (9.0, 1.15)), ((11.55, 1.15), (12.0, 1.15))]:
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops={"arrowstyle": "->"})
ax.set_xlim(-0.2, 14.7); ax.set_ylim(-0.2, 3.0); ax.axis("off")
ax.set_title("LayoutLMv3 for token classification: text, layout and image into one transformer")
ordered_blocks = blocks.sort_values("parameters")
axes[1].barh(ordered_blocks["block"], ordered_blocks["parameters"],
             color=[PALETTE["orange"] if "new" in origin else PALETTE["blue"] for origin in ordered_blocks["starts_from"]])
axes[1].set_xscale("log")
axes[1].set(xlabel="parameters (log scale)", title="Parameters per block (orange = new, starts random)")
for position, count in enumerate(ordered_blocks["parameters"]):
    axes[1].text(count * 1.15, position, f"{count:,}", va="center", fontsize=8)
axes[1].set_xlim(right=ordered_blocks["parameters"].max() * 12)
show(fig, "11_model")

**What to look for.** The list of "missing" tensors contains only `classifier.*`: the head. The bar chart has a log axis: the head is under 1 % of the parameters, yet it is the only part that knows nothing at the start.

## 6 · One batch, step by step

Before any training, push one batch through the model and look at every tensor. `B` is the batch size, `L = 512` the sequence length, `K` the number of labels.

In [ ]:
batch = collate_windows([train_set[i] for i in range(4)])
meaning = {"input_ids": "subword token ids", "attention_mask": "1 = real token, 0 = padding",
           "bbox": "box of each token on the 0..999 grid", "pixel_values": "normalised page image for the visual branch",
           "labels": "class id on the first token of each labelled box, -100 elsewhere",
           "word_index": "bookkeeping: which box a first token belongs to (-1 elsewhere); not a model input"}
display(pd.DataFrame([{"tensor": name, "shape": tuple(value.shape), "dtype": str(value.dtype).replace("torch.", ""),
                       "meaning": meaning[name]} for name, value in batch.items()]))

model = model.to(device).eval()
with torch.inference_mode():
    logits = model(**{name: batch[name].to(device) for name in MODEL_INPUTS}).logits.float().cpu()
labelled = batch["labels"] != IGNORE_INDEX
print("logits:", tuple(logits.shape), "= [B, L, K]: one score per label for every text position")
print(f"positions with a label: {int(labelled.sum())} of {labelled.numel()} "
      f"({int(batch['attention_mask'].sum())} real tokens, the rest padding)")
print(inspect.getsource(masked_cross_entropy))
by_hand = -torch.log_softmax(logits[labelled], dim=-1)[torch.arange(int(labelled.sum())), batch["labels"][labelled]].mean()
print(f"loss by hand {float(by_hand):.4f} | masked_cross_entropy {float(masked_cross_entropy(logits, batch['labels'])):.4f} "
      f"| ln(K) = {math.log(len(labels)):.4f}")
first = int(torch.nonzero(labelled[0])[1])
probabilities = logits[0, first].softmax(-1)
print(f"Untrained head, second labelled box of the first page: true label {labels[int(batch['labels'][0, first])]!r}, "
      f"model's top guess {labels[int(probabilities.argmax())]!r} with probability {float(probabilities.max()):.3f}")

**How to read this.**

- **Softmax** turns the `K` logits of one position into `K` probabilities: each logit is exponentiated and divided by the sum of all `K` exponentials. The results are positive and add up to 1, and the largest logit gets the largest probability.
- **Cross-entropy** is the loss: for each labelled position it is `-log(probability the model gave to the true label)`. A confident right answer costs almost 0; a confident wrong answer costs a lot.
- **Masked** means positions labelled `-100` are skipped: padding, non-first subwords, special tokens, and the ambiguous boxes from section 4.2.
- With a random head every label is about equally likely, so the loss should be near `ln(K)`. The three numbers printed above should agree. If the starting loss were far from `ln(K)`, something would be wrong with the labels before training even began. This is a cheap and useful check.

## 7 · Training

**What one epoch does.** The training pages are shuffled and fed in batches. For each batch: forward pass → masked cross-entropy → backward pass (gradients). After every `accumulation_steps` batches the accumulated gradient is clipped to length 1, the optimiser **AdamW** changes the weights a little, with weight decay, and the learning rate moves along its schedule. These terms, and gradient accumulation, are defined under the recipe table in section 2. After the epoch the model labels the validation pages, and the score that selects the checkpoint, validation **entity F1** (also defined in section 2), is computed. The best epoch so far is saved. If the score has not improved for `patience` epochs, training stops early.

**Concept: learning-rate schedule.** The learning rate is the step size. It *warms up* linearly from near zero over the first 10 % of updates, so the random head does not wreck the pretrained weights with huge early gradients, then *decays* linearly to zero, so the last updates only fine-adjust.

The loop is `train()` in `pipelines/documents/lineitems_training.py`. The next cell prints the effective configuration and draws the schedule this run will follow.

In [ ]:
updates_per_epoch = math.ceil(math.ceil(len(train_set) / config.batch_size) / config.accumulation_steps)
total_updates = updates_per_epoch * config.epochs
warmup_updates = max(1, round(config.warmup_fraction * total_updates))
display(pd.Series({**{k: v for k, v in vars(config).items()}, "effective batch size": config.effective_batch_size,
                   "training windows": len(train_set), "updates per epoch": updates_per_epoch,
                   "total updates": total_updates, "warmup updates": warmup_updates}, name="effective configuration (proposed values)").to_frame())
print(inspect.getsource(lr_factor))
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot([config.learning_rate * lr_factor(u, total_updates, warmup_updates) for u in range(total_updates)], color=PALETTE["blue"])
ax.axvline(warmup_updates, color=PALETTE["vermillion"], linestyle="--", label=f"end of warmup (update {warmup_updates})")
ax.set(xlabel="optimiser update", ylabel="learning rate", title=f"Planned learning rate: {total_updates} updates if all {config.epochs} epochs run")
ax.legend()
show(fig, "12_learning_rate_schedule")

### Run the training

The next cell trains (or, with `RELOAD_RUN_DIR` set, skips training). Each epoch prints one line: training loss, validation loss, validation entity F1 and field micro-F1, the learning rate and the time. `<- best so far` marks an epoch whose weights were saved.

What to expect on this dataset: the training loss should fall steeply in the first epochs (from about `ln(K)`), and entity F1 should rise from near 0 to near 1 within a few epochs, so early stopping usually ends the run before epoch 20. In the reference run on an Apple M4 Max, with a second training job sharing the GPU, an epoch took 76 to 132 seconds (14 epochs in 22 minutes). With the GPU to itself it should take roughly half of that; a CPU is several times slower. Interrupting the kernel is safe: the best epoch so far stays on disk. A repeat run with the same seed will differ slightly, because GPU arithmetic is not exactly reproducible.

In [ ]:
validation_alignments = {p.image_id: alignments[p.image_id] for p in validation_pages}
used_alignments = {p.image_id: alignments[p.image_id] for p in train_pages + validation_pages}
score = lambda predictions: validation_report(validation_alignments, predictions, annotations)

if RELOAD_RUN_DIR is None:
    context = describe_training_data(cache, used_alignments, POLICY, labels)
    context["quick_run"] = QUICK_RUN
    run_record = train(model, processor, train_set, validation_set, config, labels=labels, score=score,
                       run_dir=RUN_DIR, evaluation_dir=EVALUATION_DIR, context=context)
    print(f"\nRun {run_record['status']}: {run_record['completed_epochs']} epochs in {run_record['seconds'] / 60:.1f} min, "
          f"best epoch {run_record['best_epoch']} with validation entity F1 {run_record['best_validation']['entity_f1']:.4f}")
else:
    print("Training skipped: evaluating the saved run", RUN_DIR, "\nIts training log was:")
    for line in (RUN_DIR / "training_log.jsonl").read_text().splitlines():
        e = json.loads(line)
        print(f"epoch {e['epoch']:>2}  train loss {e['train_loss']:.4f}  val loss {e['validation_loss']:.4f}  "
              f"val entity F1 {e['validation_entity_f1']:.4f}  field micro-F1 {e['validation_field_micro_f1']:.4f}  "
              f"lr {e['learning_rate']:.2e}  {e['seconds']:.0f}s")
if QUICK_RUN:
    print("QUICK RUN: these numbers check the plumbing only. They are not a model result.")

## 8 · Curves, validation scores and error analysis

Every number in this section is **synthetic validation evidence**: it is measured on the validation split of the synthetic dataset, which shares its single page layout and its three handwriting fonts with the training split.

### 8.1 Learning curves

In [ ]:
history = pd.DataFrame(json.loads((RUN_DIR / "history.json").read_text()))
best_epoch = int(history.loc[history.validation_entity_f1.idxmax(), "epoch"])
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].plot(history.epoch, history.train_loss, marker="o", label="training")
axes[0].plot(history.epoch, history.validation_loss, marker="s", label="validation")
axes[0].set_yscale("log")
axes[0].set(xlabel="epoch", ylabel="masked cross-entropy (log scale)", title="Loss")
axes[1].plot(history.epoch, history.validation_entity_f1, marker="o", label="entity F1 (selects the checkpoint)")
axes[1].plot(history.epoch, history.validation_field_micro_f1, marker="s", label="field micro-F1 (boxes)")
axes[1].plot(history.epoch, history.validation_field_macro_f1, marker="^", label="field macro-F1 (boxes)")
axes[1].set(xlabel="epoch", ylabel="score on validation", ylim=(0, 1.02), title="Validation scores")
axes[2].plot(history.epoch, history.learning_rate, marker="o", color=PALETTE["green"])
axes[2].set(xlabel="epoch", ylabel="learning rate at the end of the epoch", title="Learning rate")
from matplotlib.ticker import MaxNLocator
for ax in axes:
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
for ax in axes[:2]:
    ax.axvline(best_epoch, color=PALETTE["grey"], linestyle=":", label=f"selected epoch {best_epoch}")
    ax.legend(fontsize=8)
fig.suptitle(f"Run {RUN_ID}: {len(history)} epochs" + ("  (QUICK RUN, not a model result)" if QUICK_RUN else ""), y=1.03)
show(fig, "13_learning_curves")
display(history[["epoch", "train_loss", "validation_loss", "validation_entity_f1", "validation_field_micro_f1",
                 "validation_field_macro_f1", "seconds"]].round(4))

**How to read the curves.**

- **Loss (left, log axis).** Both curves should fall. If the training loss keeps falling while the validation loss turns upward, the model is starting to memorise the training pages (*overfitting*). In the reference run that never happens, and the validation loss even stays *below* the training loss. Two reasons: dropout (random switching-off of units) is active only while training, and the validation pages are no harder than the training pages, because both come from the same generator and layout. So a low validation loss here says the splits are alike. It does not say the model would do as well on a new layout.
- **Scores (middle).** *Micro*-F1 counts every box equally, so frequent labels dominate. *Macro*-F1 averages the labels, so a rare label that is never predicted pulls it down. A macro curve that lags behind micro means the rare labels (high row slots) are learned later or not at all.
- The dotted line is the selected epoch: the **first** epoch that reached the best entity F1. Later epochs with the same score do not replace it. Selection uses validation, so the validation score of the selected epoch is slightly optimistic. That is what the untouched test split is for.

### 8.2 Reload the saved run and score it

From here on the notebook uses the **saved best checkpoint**, reloaded from disk. That makes this section work after a kernel restart (set `RELOAD_RUN_DIR`), and it also proves the saved weights load. `evaluate_pages` aligns, labels and scores the validation pages in one call; the opt-in final evaluation calls the same function on the test pages.

In [ ]:
model, processor, saved_labels, run_record = load_run(RUN_DIR, device)
assert saved_labels == labels, "the saved run uses another label list"
evaluation = evaluate_pages(model, processor, validation_pages, annotations, POLICY, labels, device=device,
                            batch_size=config.eval_batch_size)
report, predictions = evaluation["report"], evaluation["predictions"]
(EVALUATION_DIR / "validation_report.json").write_text(json.dumps({"split": "validation", "provenance": "synthetic",
                                                                   "quick_run": QUICK_RUN, **report}, indent=2))
scored = pd.DataFrame(box_rows(evaluation["alignments"], predictions))
boxes, entities = report["boxes"], report["entities"]
print("SYNTHETIC VALIDATION EVIDENCE" + (" · QUICK RUN, not a model result" if QUICK_RUN else ""), "· run", RUN_ID,
      "· epoch", run_record.get("best_epoch"))
print(f"Pages: {report['pages']} ({report['pages_unreadable']} unreadable for M4, kept in the row counts as failures)")
print(f"Boxes: {boxes['boxes']:,} | scored {boxes['scored_boxes']:,} | masked, not scored {boxes['masked_boxes']}")
headline = pd.DataFrame([
    {"score": "field labels, micro (boxes)", **boxes["field_micro"]},
    {"score": "O (static text and handwriting), reported apart", **boxes["outside"]},
    {"score": "entities (a field with exactly its boxes)", **{k: entities[k] for k in boxes["field_micro"]}}])
display(headline.rename(columns={"true_positive": "correct", "support": "true"}).round(4))
print(f"Field macro-F1 over the {boxes['field_labels_with_support']} labels that occur in validation: {boxes['field_macro_f1']:.4f}")
print(f"Accuracy over all scored boxes (O included): {boxes['accuracy_all_scored']:.4f}")
print(f"Ground-truth fields with no usable M4 box (outside every recall above): {entities['fields_without_box']}")
multi_box_fields = sum(len(match.box_ids) > 1 for a in evaluation["alignments"].values() for match in a.fields
                       if match.status == "matched")
print(f"Validation fields carried by more than one box: {multi_box_fields} of {entities['support']:,}"
      + (" → every entity is one box: the entity row cannot say more than the field-box row" if not multi_box_fields else ""))

**How to read the headline table.**

- **Precision** = of the boxes the model gave this label, the share that truly have it. **Recall** = of the boxes that truly have the label, the share the model found. **F1** is their harmonic mean. `correct`, `predicted` and `true` are the counts behind them.
- **Field labels, micro** covers only the boxes that are printed field values. `O` has its own row, because `O` is half of all boxes and easy; mixing it in would flatter the result.
- **Entities** is the score that selected the checkpoint (defined in section 2): a field counts only when its label **and** its whole set of boxes are right. Do not read this row as a stricter second opinion. The last printed line says why: this validation split has **no** field carried by two boxes, so every entity is one box, and the entity row is in effect the field-box row again. In the reference run both read 2,128 of 2,128. The case an entity score exists for, a description wrapped over two boxes, does not occur in validation, so how the model handles it is **unmeasured**.
- The last line is the reminder from section 4: fields M4 never delivered are in **no** recall above. A recall of 1.0 here would still leave most quantities missing.
- **A score of 1.0 can happen here, and it did in the reference run.** It means that labelling the boxes of *this* generator is an easy task that 350 pages are enough to solve: every validation page has the same layout as the training pages. It does not mean the model is finished. The next cell measures how easy the task is, and section 8.9 shows how quickly the same model fails when the page is moved.

#### How easy is this task? A baseline with no model

Before reading a score as an achievement, ask what the simplest possible method would score. Every page of this dataset has the same layout, so try **position alone**: for each validation box, find the training box whose four coordinates (`box_norm`) are closest and copy its label. This method reads no text, looks at no image and has no trained weight. It is scored by the same functions as the model, on the same boxes. The helper is `position_baseline` in `pipelines/documents/lineitems_eval.py`: a nearest-neighbour lookup in a dozen lines of NumPy.

In [ ]:
baseline_predictions = position_baseline(train_pages, alignments, validation_pages)   # four coordinates per box, nothing else
baseline_rows = pd.DataFrame(box_rows(evaluation["alignments"], baseline_predictions))

def boxes_right(name, table):
    table = table[table.scored]
    fields, outside = table[table.true != "O"], table[table.true == "O"]
    return {"method": name, "scored boxes right": share(int(table.correct.sum()), len(table)),
            "field boxes right": share(int(fields.correct.sum()), len(fields)),
            "O boxes right": share(int(outside.correct.sum()), len(outside))}

display(pd.DataFrame([boxes_right("nearest training box by position: no text, no image, no model", baseline_rows),
                      boxes_right("LayoutLMv3, this run" + (" · QUICK RUN" if QUICK_RUN else ""), scored)]).set_index("method"))
baseline_wrong = baseline_rows[baseline_rows.scored & ~baseline_rows.correct]
handwritten = baseline_rows[baseline_rows.reason == "handwriting"]
print("Boxes the position lookup gets wrong, by their true field type:", {name: int(n) for name, n in baseline_wrong.true_type.value_counts().items()})
print("Handwritten boxes the position lookup leaves as O:", share(int(handwritten.correct.sum()), len(handwritten)))
write_json_atomic(EVALUATION_DIR / "position_baseline_validation.json", {
    "split": "validation", "provenance": "synthetic", "quick_run": QUICK_RUN,
    "method": "label of the nearest training box by box_norm (x0, y0, x1, y1); no text, no image, no model",
    **box_summary(baseline_rows.to_dict("records"))})

**What to look for.** Compare the two rows. In the reference run the position lookup is right for 4,285 of the 4,324 scored boxes: 2,089 of 2,128 field boxes and all 2,196 `O` boxes, the 70 handwritten boxes included. The trained model's 4,324 of 4,324 is therefore 39 boxes better than a method that cannot read. Of those 39 boxes, 37 are page fields (values outside the parts table), one is an amount and one is a remark.

Two lessons follow.

1. **The high score mostly measures the layout.** On this generator a box's place on the page nearly decides its label. A model can reach a very high score here without understanding a table, and the score alone cannot tell you whether it does.
2. **A method that relies on position must fail when the page moves.** Section 8.9 moves the pages to find out how much the trained model behaves like this lookup.

### 8.3 Scores per field type and per label

In [ ]:
per_type = pd.DataFrame(type_metrics(box_rows(evaluation["alignments"], predictions))).rename(
    columns={"label": "field type", "true_positive": "correct", "support": "true boxes"})
display(per_type.round(4))
per_label = pd.DataFrame(report["per_label"]).rename(columns={"true_positive": "correct", "support": "true boxes"})
occurring = per_label[(per_label["true boxes"] > 0) & (per_label.label != "O")]
print(f"{len(occurring)} field labels occur in validation; {int((occurring.f1 == 1).sum())} of them have F1 = 1.0.")
print("The weakest labels (lowest F1 first):")
display(occurring.sort_values(["f1", "true boxes"]).head(12).round(4))
never_true = per_label[(per_label["true boxes"] == 0) & (per_label.predicted > 0)]
print("Labels the model predicted although no validation box has them:", "none" if never_true.empty else "")
if not never_true.empty:
    display(never_true)

fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(len(per_type))
for offset, column, colour in ((-0.2, "precision", PALETTE["blue"]), (0.2, "recall", PALETTE["orange"])):
    bars = ax.bar(x + offset, per_type[column].fillna(0), width=0.4, label=column, color=colour)
    ax.bar_label(bars, labels=["n/a" if pd.isna(v) else f"{v:.3f}" for v in per_type[column]], fontsize=7, rotation=90, padding=2)
ax.set_xticks(x, [f"{name}\n(n={n})" for name, n in zip(per_type["field type"], per_type["true boxes"])])
ax.set(ylim=(0, 1.22), ylabel="score on validation boxes", title="Per field type (row slot ignored): precision and recall, n = true boxes")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, fontsize=8)
show(fig, "14_per_type_scores")

**What to look for.** The per-type table ignores the row slot and the `B-`/`I-` prefix: it answers "did the model recognise an amount as an amount?". The per-label list is stricter: it needs the right slot too, so `part_7_amount` predicted as `part_6_amount` is wrong there and right in the per-type table. Quantity has a small `n` because most quantity cells never reached M5. Read a score with a small `n` as an anecdote, not as a rate.

### 8.4 Two confusion views

A **confusion matrix** counts, for every true class (rows), what the model predicted (columns). Everything off the diagonal is an error. Colour shows the share of the row; the number is the count.

- **Field type × field type**: which kinds of text get confused.
- **True row slot × predicted row slot**: for table cells only, does the model put the cell in the right row?

In [ ]:
type_matrix, type_names = type_confusion(box_rows(evaluation["alignments"], predictions))
slot_matrix, slot_names = slot_confusion(box_rows(evaluation["alignments"], predictions))
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
plot_confusion(axes[0], type_matrix, type_names, type_names, title="Field type: true (rows) against predicted (columns)",
               xlabel="predicted field type", ylabel="true field type")
plot_confusion(axes[1], slot_matrix, slot_names[:-1], slot_names, title="Row slot of table cells: true against predicted",
               xlabel="predicted row slot", ylabel="true row slot")
fig.suptitle("Validation boxes" + ("  (QUICK RUN)" if QUICK_RUN else ""), y=1.0)
show(fig, "15_confusion_matrices")
off_type = int(type_matrix.sum() - np.trace(type_matrix))
off_slot = int(slot_matrix.sum() - np.trace(slot_matrix[:, :-1]))
print(f"Field-type errors: {share(off_type, int(type_matrix.sum()))} scored boxes")
print(f"Row-slot errors among table cells: {share(off_slot, int(slot_matrix.sum()))} cells "
      f"({int(slot_matrix[:, -1].sum())} of them were given no row at all)")

**What to look for.** In the left matrix, look at the `O` row and the `O` column: a field read as `O` is a *missed* field; `O` read as a field is an *invented* one (for example handwriting taken for a printed remark). In the right matrix, errors next to the diagonal mean "right kind of cell, neighbouring row". Those would hurt most on a layout where rows sit elsewhere.

### 8.5 Row-level results on estimate pages

What M5 must finally deliver is a **row**. For each ground-truth table row: are its description, action, quantity and amount boxes all labelled with the right field **and** the right slot? A field is *right* when the boxes predicted `part_N_<field>` are exactly the boxes aligned to it. Two counts are reported, both over **all** ground-truth rows, including the rows of pages M4 could not read:

- **complete (delivered fields)**: every core field M4 delivered is right, and no stray box was given this row's labels;
- **complete (all four)**: as above, and M4 delivered all four fields. The missing quantity boxes cap this one.

In [ ]:
rows_table = pd.DataFrame(row_rows(evaluation["alignments"], predictions, annotations))
row_counts = report["rows"]
total_rows = row_counts["rows"]
print(f"Ground-truth rows on validation estimate pages: {total_rows} ({row_counts['rows_on_unreadable_pages']} on pages M4 could not read)")
print("Complete row labelling, delivered fields:", share(row_counts["complete_available"], total_rows))
print("Complete row labelling, all four fields :", share(row_counts["complete_all_four"], total_rows),
      f"| rows where M4 delivered all four: {row_counts['rows_with_all_four_delivered']}")
per_field = pd.DataFrame([{"field": kind, "ground-truth rows": total_rows, "delivered by M4": row_counts[f"{kind}_delivered"],
                           "labelled right": row_counts[f"{kind}_right"]} for kind in CORE_ROW_FIELDS]).set_index("field")
display(per_field)
fig, ax = plt.subplots(figsize=(8.5, 3.4))
x = np.arange(len(per_field))
for offset, column, colour in ((-0.27, "ground-truth rows", PALETTE["grey"]), (0, "delivered by M4", PALETTE["sky"]),
                               (0.27, "labelled right", PALETTE["blue"])):
    bars = ax.bar(x + offset, per_field[column], width=0.27, label=column, color=colour)
    ax.bar_label(bars, fontsize=8)
ax.set_xticks(x, per_field.index)
ax.set(ylabel="rows (validation)", title="From ground-truth rows to correctly labelled cells: where rows are lost")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3, fontsize=8)
show(fig, "16_row_level")

**What to look for.** Each group of three bars is a funnel. The drop from grey to light blue is lost **before** M5 (OCR did not deliver the cell, or M4 refused the page). The drop from light blue to dark blue is M5's own error. For quantity the first drop dominates: it is an M4 finding, and no amount of M5 training can close it.

### 8.6 Can the model's confidence be trusted?

The model's *confidence* for a box is the probability of its chosen label (softmax of the logits at the first token). Two pictures:

- a histogram of confidence for correct and for wrong boxes;
- a **reliability plot**: boxes are grouped by confidence; for each group the share that is actually correct is plotted. A perfectly *calibrated* model lies on the diagonal (of the boxes it is 80 % sure about, 80 % are right).

**Concept: expected calibration error (ECE).** One number for the reliability plot: in each confidence group take the gap between the mean confidence and the share that is correct, then average the gaps, weighting each group by its number of boxes. 0 means perfectly calibrated on these boxes.

In [ ]:
scored_only = scored[scored.scored]
table = pd.DataFrame(reliability(scored.to_dict("records"), bins=10))
ece = expected_calibration_error(table.to_dict("records"))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
bins = np.linspace(0, 1, 21)
axes[0].hist(scored_only[scored_only.correct].confidence, bins=bins, color=PALETTE["blue"], alpha=0.75,
             label=f"correct ({int(scored_only.correct.sum()):,})")
axes[0].hist(scored_only[~scored_only.correct].confidence, bins=bins, color=PALETTE["vermillion"], alpha=0.85,
             label=f"wrong ({int((~scored_only.correct).sum()):,})")
axes[0].set_yscale("log")
axes[0].axvline(MIN_CONFIDENCE, color="black", linestyle="--", label=f"adapter threshold {MIN_CONFIDENCE} (proposed)")
axes[0].set(xlabel="model confidence of the predicted label", ylabel="scored boxes (log scale)", title="Confidence of correct and wrong boxes")
axes[0].legend(loc="upper left", fontsize=8)
filled = table[table.boxes > 0]
axes[1].plot([0, 1], [0, 1], color=PALETTE["grey"], linestyle=":", label="perfect calibration")
axes[1].plot(filled.mean_confidence, filled.accuracy, marker="o", color=PALETTE["blue"], label="this model")
for _, row in filled.iterrows():
    axes[1].annotate(f"n={int(row.boxes)}", (row.mean_confidence, row.accuracy), textcoords="offset points", xytext=(4, -10), fontsize=7)
axes[1].set(xlabel="mean confidence in the bin", ylabel="share of boxes that are correct", xlim=(0, 1.02), ylim=(0, 1.05),
            title=f"Reliability on validation (expected calibration error {ece:.4f})")
axes[1].legend(loc="lower right", fontsize=8)
show(fig, "17_confidence")
wrong = scored_only[~scored_only.correct]
low_but_right = scored_only[scored_only.correct & (scored_only.confidence < MIN_CONFIDENCE)]
print(f"Wrong boxes: {len(wrong)} | of them below the {MIN_CONFIDENCE} threshold: {int((wrong.confidence < MIN_CONFIDENCE).sum())}",
      f"| correct boxes below it: {len(low_but_right)}")
if len(low_but_right):
    print("Correct boxes the adapter would mark uncertain, because the model's confidence is below the threshold:")
    display(low_but_right[["image_id", "text", "true", "confidence", "ocr_confidence"]].round(3))

**How to read this.** The scores are **uncalibrated**: nothing in training forces "0.9" to mean "right nine times in ten".

**With no wrong box, these two pictures show very little, and the reference run has none.** The histogram then has no red bars. Every filled group of the reliability plot sits at "100 % correct", so the curve is a flat line along the top, and the calibration error only measures how far the confidences are below 1. A reliability plot needs mistakes before it can show over-confidence. Do not read the flat line as "well calibrated": it is the picture of a task with no errors to calibrate on.

**The one fact the cell does show** is in its last lines: boxes that are labelled **correctly** but with a confidence below the adapter's threshold of 0.5 (a proposed value). The serving adapter marks a field *uncertain* below that threshold and then does not accept its value. In the reference run there are 3 such boxes, listed in the table: all three are quantity cells (`B-part_3_quantity`, OCR text `1`, model confidence about 0.34). So the threshold withholds three quantities that were labelled and read correctly. Section 10.1 meets them again as null quantities with the reason `model_or_ocr_confidence_low`. A plausible cause is that quantity is the table-cell label with the fewest training boxes (150 on 350 pages); this notebook does not test that explanation.

Treat confidence as a weak hint for the reviewer, never as a guarantee, and remember that it describes the *label*, not whether OCR read the text correctly.

### 8.7 Errors by surveyor mark, and what happens to handwriting

Does ink on a row make its cells harder to **label**? Section 4.4 showed ink makes amounts harder to *read*; this is the other question. Each row of the table is one kind of mark on the row.

The cell first checks the rule that matters most for safety: **a handwritten replacement amount must never be taken for the printed amount.** It counts how the model labelled the boxes that are surveyor handwriting, and the masked boxes in which print and handwriting are merged (those have no training label, but in serving the model still has to say something about them).

In [ ]:
handwriting = scored[scored.reason == "handwriting"]
merged = scored[scored.reason == "print_and_handwriting"]
counts = lambda table: {name: int(n) for name, n in table.pred_type.value_counts().items()}
print("Handwritten boxes (true label O):", len(handwriting), "| the model's label types:", counts(handwriting))
print("   of them given a printed-AMOUNT label:", share(int((handwriting.pred_type == "amount").sum()), len(handwriting)))
print("Masked boxes, print merged with handwriting:", len(merged), "| the model's label types:", counts(merged))
print("   of them given a printed-AMOUNT label:", share(int((merged.pred_type == "amount").sum()), len(merged)))
inside_merged = Counter(field_type(box.field) for a in evaluation["alignments"].values() for box in a.boxes
                        if box.reason == "print_and_handwriting")
print("   the printed field inside them, from the alignment:", dict(inside_merged))
print(f"   so the model called {share(int((merged.pred_type == 'O').sum()), len(merged))} merged boxes O, "
      f"and gave {int((merged.pred_type == 'remarks').sum())} the label of a printed remark")

cells = scored[scored.scored & scored.true_slot.notna() & (scored.page_type == "repair_estimate")].copy()
style_of = {(r.image_id, r.slot): r.row_style for r in rows_table.itertuples()}
cells["row_style"] = [style_of.get((image_id, int(slot)), "unmarked") for image_id, slot in zip(cells.image_id, cells.true_slot)]
by_style = cells.groupby("row_style").agg(cells=("correct", "size"), cells_right=("correct", "sum"))
row_style = rows_table.groupby("row_style").agg(rows=("slot", "size"), complete_delivered=("complete_available", "sum"))
by_style = by_style.join(row_style, how="outer").fillna(0).astype(int)
by_style["cell accuracy"] = (by_style.cells_right / by_style.cells.where(by_style.cells > 0)).round(4)
by_style = by_style.sort_values("cell accuracy")
display(by_style)
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.barh(by_style.index, by_style["cell accuracy"].fillna(0), color=PALETTE["blue"])
for i, (name, row) in enumerate(by_style.iterrows()):
    ax.text(0.02, i, f"{int(row.cells_right)} of {int(row.cells)} cells; rows complete {int(row.complete_delivered)} of {int(row.rows)}",
            va="center", fontsize=8, color="white" if row.cells else "black")
ax.set(xlim=(0, 1.0), xlabel="share of delivered table cells labelled exactly right (validation)", ylabel="surveyor mark on the row",
       title="Labelling accuracy of table cells, by the mark on their row")
show(fig, "18_errors_by_mark_style")

**What to look for.** The first lines should show **0** handwritten boxes labelled as a printed amount. If that number is ever above 0, a surveyor's revised figure could reach the declared-amount field, and the model must not be served. A merged box holds a printed remark **and** handwriting in one OCR line, so two answers are defensible, and the printout says which one the model gives. In the reference run it calls 31 of the 32 merged boxes `O` and gives one the label of the printed remark: it treats a merged box as handwriting, not as the remark. The cost is small and visible. The printed remark in such a box is not attached to its row, and the box stays in the unparsed evidence. `adapt_predictions` builds no value from a remark, so no amount, quantity or operation changes either way. In the table, compare the marked rows with `unmarked` and `tick`. The counts per style are small on 76 validation pages, so a difference of one or two cells is not a finding. `rows complete` counts rows on unreadable pages as not complete.

### 8.8 The pages with the most errors

The last picture shows the validation pages with the most wrongly labelled boxes (ties broken by lowest confidence). Correct field boxes are thin blue, correct `O` boxes are not drawn, wrong boxes are thick red with `true → predicted`, masked boxes are yellow.

In [ ]:
page_errors = (scored[scored.scored].groupby("image_id").agg(errors=("correct", lambda c: int((~c).sum())),
                                                             min_confidence=("confidence", "min"))
               .sort_values(["errors", "min_confidence"], ascending=[False, True]))
worst = list(page_errors.index[:3])
estimates_only = all(cache.by_id(image_id).page_type == "repair_estimate" for image_id in worst)
fig, axes = plt.subplots(1, len(worst), figsize=(6.2 * len(worst), 5.7 if estimates_only else 9.0), squeeze=False)
for ax, image_id in zip(axes[0], worst):
    page_rows = scored[scored.image_id == image_id]
    by_id = {b.box_id: b for b in evaluation["alignments"][image_id].boxes}
    drawn, colours, texts, widths = [], [], [], []
    for r in page_rows.itertuples():
        if not r.scored:
            style = (TYPE_COLOURS["masked"], f"masked, predicted {r.pred}", 1.6)
        elif not r.correct:
            style = (PALETTE["vermillion"], f"{r.true} → {r.pred} ({r.confidence:.2f})", 2.6)
        elif r.true != "O":
            style = (PALETTE["blue"], None, 0.9)
        else:
            continue
        drawn.append(by_id[r.box_id].box_px); colours.append(style[0]); texts.append(style[1]); widths.append(style[2])
    image = Image.open(cache.by_id(image_id).rectified_path).convert("RGB")
    draw_boxes(ax, image, drawn, colours, texts=texts, text_size=6,
               extent=(40, 150, 1200, 1130) if cache.by_id(image_id).page_type == "repair_estimate" else None)
    for patch, width in zip(ax.patches, widths):
        patch.set_linewidth(width)
    ax.set_title(f"{image_id}: {int(page_errors.loc[image_id, 'errors'])} wrong of {int(page_rows.scored.sum())} scored boxes", fontsize=9)
fig.suptitle("Validation pages with the most labelling errors (red = wrong: true → predicted)" if page_errors.errors.sum() else
             "No labelling error on validation: the three pages where the model was least sure", y=1.0)
show(fig, "19_worst_pages")
display(scored[scored.scored & ~scored.correct][["image_id", "text", "true", "pred", "confidence", "ocr_confidence"]].head(15))
print(f"Pages with at least one wrong box: {share(int((page_errors.errors > 0).sum()), len(page_errors))} readable validation pages")

**What to look for.** Read the red labels: is the *type* wrong, or only the *slot*? Is the text itself damaged (`ocr_confidence` low, odd characters), which would make the box hard for anyone? If no box is red, the pictures show the pages where the model was least sure. Remember what cannot appear here at all: cells that OCR never delivered.

### 8.9 A stress probe: the same pages, moved

Every page of this dataset has its parts table at exactly the same place. So a high score cannot tell two models apart: one that understands "this is the third row of the table", and one that has memorised "a box at height 0.38 is row 3". Section 4.3 named this as the weakness of row-slot labels, and the position lookup of section 8.2 showed how far position alone goes. Here it is measured for the trained model.

The probe takes each readable validation estimate page and **moves its whole content** by a number of pixels: the image is shifted and every box coordinate is shifted by the same amount. Nothing else changes: same text, same boxes, same true labels. The page is moved **down, up, right and left**, because a real page can be off in any direction. A table row on this layout is about 79 pixels high. For every move the cell counts, among the table-cell boxes:

- how many still get the right **field type**, and how many the right **row slot**;
- how many get a row number exactly **one too high** or **one too low**. If nearly all wrong cells are off by the same one row, the rows were not scrambled: they were renumbered, as a lookup by height would do.

This is a synthetic stress test on validation pages. It is not an evaluation on a second layout, and it selects nothing.

In [ ]:
ROW_PITCH = 79                                              # pixels between two table rows on this layout
VERTICAL = [-79, -40, -20, -10, 0, 10, 20, 40, 79, 158]     # pixels the content is moved down (negative: up)
SIDEWAYS = [-40, -20, -10, 10, 20, 40]                      # pixels the content is moved right (negative: left)
probe_pages = [p for p in validation_pages if p.page_type == "repair_estimate" and p.page_status != "unreadable"]

def labels_of_moved_page(page, dx, dy):
    record, image = page.page, Image.open(page.rectified_path).convert("RGB")
    moved = Image.new("RGB", image.size, image.getpixel((5, 5)))     # paper colour where the content moved away
    moved.paste(image, (dx, dy))
    base = page_words(record)
    step_x, step_y = dx / record.transform.corrected_width, dy / record.transform.corrected_height
    moved_words = PageWords(base.boxes, base.readable, base.texts, tuple(
        layout_bbox((b.box_norm[0] + step_x, b.box_norm[1] + step_y, b.box_norm[2] + step_x, b.box_norm[3] + step_y))
        for b in (base.boxes[i] for i in base.readable)))
    encoding = encode_page(processor, moved, moved_words)
    assert encoding["input_ids"].shape[0] == 1, "the probe assumes one window per page"
    with torch.inference_mode():
        best = model(**{k: encoding[k].to(device) for k in MODEL_INPUTS}).logits[0].argmax(-1).cpu()
    found = {}
    for token, word in enumerate(encoding.word_ids(0)):
        if word is not None and word not in found and int(encoding["offset_mapping"][0, token, 0]) == 0:
            found[base.readable[word]] = labels[int(best[token])]
    return found                                                     # position of the box in reading order -> predicted label

def move_name(dx, dy):
    if not (dx or dy):
        return "not moved"
    return f"{'down' if dy > 0 else 'up'} {abs(dy)} px" if dy else f"{'right' if dx > 0 else 'left'} {abs(dx)} px"

probe, kept_by_slot = [], {}
for dx, dy in [(0, dy) for dy in VERTICAL] + [(dx, 0) for dx in SIDEWAYS]:
    tally, no_row, kept = Counter(), Counter(), Counter()
    for page in probe_pages:
        predicted = labels_of_moved_page(page, dx, dy)
        for position, box in enumerate(evaluation["alignments"][page.image_id].boxes):
            if box.outcome == "field" and row_slot(box.field) is not None:
                guess = field_name(predicted[position])
                true_slot, slot = row_slot(box.field), row_slot(guess)
                tally["cells"] += 1
                tally["type"] += field_type(guess) == field_type(box.field)
                tally["slot"] += slot == true_slot
                tally["higher"] += slot == true_slot + 1
                tally["lower"] += slot == true_slot - 1
                kept[true_slot] += slot == true_slot
                if slot is None:
                    no_row[f"{field_type(box.field)} → {field_type(guess)}"] += 1     # true type → what it was labelled instead
    kept_by_slot[(dx, dy)] = kept
    probe.append({"move": move_name(dx, dy), "dx (pixels)": dx, "dy (pixels)": dy, "dy (rows)": round(dy / ROW_PITCH, 2),
                  "table cells": tally["cells"], "field type right": tally["type"], "row slot right": tally["slot"],
                  "one row too high": tally["higher"], "one row too low": tally["lower"], "given no row": sum(no_row.values()),
                  "cells given no row": ", ".join(f"{count} {name}" for name, count in no_row.most_common()) or "-"})
probe = pd.DataFrame(probe)
display(probe.set_index("move").drop(columns=["dx (pixels)"]))

cells = probe["table cells"]
vertical, sideways = probe[probe["dx (pixels)"] == 0], probe[probe["dx (pixels)"] != 0]
fig, axes = plt.subplots(1, 2, figsize=(14, 3.9), gridspec_kw={"width_ratios": [1.6, 1]})
ax = axes[0]
ax.plot(vertical["dy (pixels)"], vertical["field type right"] / cells[vertical.index], marker="o", label="field type right (description, amount, …)")
ax.plot(vertical["dy (pixels)"], vertical["row slot right"] / cells[vertical.index], marker="s", label="row slot right (part_N)")
ax.plot(vertical["dy (pixels)"], (vertical["one row too high"] + vertical["one row too low"]) / cells[vertical.index], marker="^",
        linestyle="--", color=PALETTE["grey"], label="row number off by exactly one row")
for edge in (-ROW_PITCH, ROW_PITCH):
    ax.axvline(edge, color=PALETTE["black"], linestyle=":", linewidth=1, label="one row height" if edge > 0 else None)
ax.set(xlabel="page content moved by (pixels): negative = up, positive = down", ylabel="share of table-cell boxes", ylim=(0, 1.05),
       title="Moved up or down")
ax.legend(loc="center right", fontsize=8)
x = np.arange(len(sideways))
for offset, column, colour in ((-0.2, "field type right", PALETTE["blue"]), (0.2, "row slot right", PALETTE["vermillion"])):
    bars = axes[1].bar(x + offset, sideways[column] / cells[sideways.index], width=0.4, color=colour, label=column)
    axes[1].bar_label(bars, labels=[str(v) for v in sideways[column]], fontsize=7, rotation=90, padding=2)
axes[1].set_xticks(x, sideways["move"], rotation=30, ha="right")
axes[1].set(ylabel="share of table-cell boxes", ylim=(0, 1.19), title=f"Moved sideways (cells right, of {int(cells.iloc[0])})")
axes[1].legend(loc="lower center", ncol=2, fontsize=8)
fig.suptitle(f"Stress probe on {len(probe_pages)} validation estimate pages: the same pages, moved"
             + (" (QUICK RUN)" if QUICK_RUN else ""), y=1.03)
show(fig, "19b_shift_probe")
probe.to_json(EVALUATION_DIR / "shift_probe_validation.json", orient="records", indent=2)

at = lambda dx, dy: probe[(probe["dx (pixels)"] == dx) & (probe["dy (pixels)"] == dy)].iloc[0]
down, up, highest = at(0, ROW_PITCH), at(0, -ROW_PITCH), max(kept_by_slot[(0, 0)])
print(f"One row down: {share(int(down['row slot right']), int(down['table cells']))} cells keep their row number, "
      f"{int(down['one row too high'])} get the next number. Of those that keep it, {kept_by_slot[(0, ROW_PITCH)][highest]} "
      f"are cells of row {highest}, the highest number on these pages.")
print(f"One row up  : {share(int(up['row slot right']), int(up['table cells']))} cells keep their row number, "
      f"{int(up['one row too low'])} get the number before. Of those that keep it, {kept_by_slot[(0, -ROW_PITCH)][1]} "
      "are cells of row 1, which has no lower number.")

**How to read the probe.** The row "not moved" repeats the validation result. In the left plot, 0 is the unmoved page, the right half moves the content down and the left half moves it up.

- If the **row slot** line falls while the **field type** line holds, the model knows what a cell *is* from its text and column, but takes the row number from where the cell sits on the page. If both lines stayed flat, the model would be using the table's own structure.
- The dashed grey line counts the cells that are off by **exactly one row**. Where it is high, the rows were renumbered consistently, all in one direction.
- The right plot shows the sideways moves. A sideways move does not change any cell's height, so it tests something else: whether the *columns* are recognised by position.

**What the reference run measured** (36 pages, 716 table cells per move).

| Move | Field type right | Row slot right | What happened to the others |
| --- | ---: | ---: | --- |
| 10 px down, up, right or left | 716 | 716 | nothing changes |
| 20 px down (a quarter of a row) | 716 | 633 | 51 one row too high, 5 one too low, 27 further off |
| 40 px down (half a row) | 716 | 311 | 106 one too high, 201 one too low, 98 further off |
| 79 px down (one row) | 716 | 19 | 697 one row too high |
| 158 px down (two rows) | 716 | 13 | 69 one too high, 6 one too low, 628 further off |
| 20 px up | 715 | 683 | 12 one too low, 12 one too high, 9 others |
| 40 px up | 715 | 253 | 419 one too high, 19 one too low, 25 others |
| 79 px up (one row) | 716 | 162 | 554 one row too low |
| 20 px right | 716 | 716 | nothing changes |
| 40 px right | 694 | 694 | 22 quantity cells labelled `O` |
| 20 px left | 713 | 713 | 3 quantity cells labelled `O` |
| 40 px left | 688 | 688 | all 28 quantity cells labelled `O` |

So, for this model on these pages:

- **It tolerates about 10 pixels** in any direction. "The page must match to the pixel" would be too strong a statement.
- **Row numbers break at a fraction of a row height, up or down.** A quarter of a row costs 83 cells when moving down and 33 when moving up. At half a row most row numbers are wrong, and they are wrong in both directions at once: between two rows the model has no consistent answer.
- **At a whole row it is consistent again, and consistently wrong.** One row down, 697 of 716 cells get the next row number. One row up, 554 get the number before. The cells that keep their number are mostly the ones that have nowhere to go: 17 of the 19 are cells of row 8, the highest number the model knows, and 133 of the 162 are cells of row 1. The row number follows the absolute height on the page, not the position in the table.
- **The field type is stable under every vertical move tested** (at least 715 of 716 cells) and **loses a few percent under a sideways move of 40 pixels** (22 and 28 of 716 cells). Every cell lost sideways is a quantity cell that becomes `O`: the narrow quantity column is recognised by where it is.

A real scan or photograph will not hold the table within ten pixels of this place. This is the strongest limit of the trained model, and the measured reason for the first next step in section 13.

## 9 · Reload a saved run without retraining

Nothing above needs to be repeated to look at a finished run again:

1. restart the kernel;
2. set `RELOAD_RUN_DIR = ROOT / "artifacts/models/lineitems/<run_id>"` in the configuration cell;
3. run the notebook from the top. Section 3.1 first checks that the run was trained with the present label policy and M4 cache, and stops if it was not. The data sections run as before (seconds), the training cell is skipped, and section 8.2 loads `best/` from that folder.

With `QUICK_RUN = True` as well, the reload uses a few pages only and writes to `artifacts/evaluation/<run_id>-quick-check/`, so the full run's reports stay as they are.

A run folder holds `best/` (weights and processor, in the layout that both transformers 4 and 5 load), `labels.json`, `config.json`, `history.json`, `training_log.jsonl` and `manifest.json` (the run record: environment, code revision, data hashes, schedule). The cell below shows the record of the run in use. This is a reload for evaluation; resuming an *interrupted* training is a different thing (`train(..., resume=True)` continues from `last.pt`).

In [ ]:
print("Run in use:", RUN_DIR)
display(pd.Series({
    "status": run_record["status"], "completed epochs": run_record.get("completed_epochs"),
    "selected epoch": run_record.get("best_epoch"), "stopped early": run_record.get("stopped_early"),
    "training time (min)": round(run_record["seconds"] / 60, 1), "device": run_record["environment"]["device"],
    "parameters": f"{run_record['parameters']:,}", "weights sha256": str(run_record.get("weights_sha256"))[:16] + "…",
    "code revision": f"{str(run_record['code_revision']['commit'])[:7]} (dirty: {run_record['code_revision']['dirty']})",
    "M4 cache manifest": run_record["context"]["m4_cache"]["manifest_sha256"][:16] + "…",
    "label policy": run_record["context"]["label_policy"]["version"],
    "quick run": run_record["context"].get("quick_run")}, name="run record").to_frame())
print("Files:", sorted(p.name for p in RUN_DIR.iterdir()))

## 10 · The serving-path check

A model that scores well in a notebook and cannot be loaded by the application is not ready for anything. This section proves the path end to end with the **application's own code**:

1. **Export** the run as a registry entry into a scratch folder (the real export in section 12 is opt-in).
2. **Load** it with `LocalLayoutLMv3`, the loader in `src/claim_cmev/documents/line_items/layoutlmv3.py`.
3. **Predict** every validation page with `LocalLayoutLMv3.predict` and confirm the labels are **identical** to the notebook's own predictions.
4. **Adapt**: pass the labels to `adapt_predictions`, which builds shared `LineItem` records through the existing vocabulary and exact-decimal parser.
5. **Score** the line items against the claim ground truth in `claims.jsonl`.

**Why the checkpoint must declare its text unit.** The loader used to accept only word boxes. A registry entry now carries `preprocessing.json` with `"text_unit_granularity": "line"`, and the loader refuses any page whose boxes are another unit, in both directions. Boxes are never split or merged to fit a model. A checkpoint without that file is treated as a word model and gets the box text exactly as OCR wrote it, so the teammate's checkpoint behaves as before.

**The loader fails closed on a damaged entry.** An entry that names its text unit must declare its whole page encoding (window length, stride, box grid, whitespace handling, label rule); nothing is filled in by default. And when the folder has a `manifest.json` that records `weights_sha256`, the weights file must have exactly that hash, or the entry is refused.

The limits written into the entry (`known_limits`) are built here from the numbers this execution measured in sections 8.2 and 8.9, so an entry can never carry a remembered figure from another run.

In [ ]:
# The limits written into the entry quote what THIS execution measured (sections 8.2 and 8.9), not a remembered number.
slots_right = lambda dx, dy: f"{int(at(dx, dy)['row slot right'])} of {int(at(dx, dy)['table cells'])}"
types_right = lambda dx, dy: f"{int(at(dx, dy)['field type right'])} of {int(at(dx, dy)['table cells'])}"
baseline_scored = baseline_rows[baseline_rows.scored]
KNOWN_LIMITS = (
    "Trained and scored on synthetic pages only: one page layout, three handwriting fonts, no photograph, no real estimate. "
    f"Measured position dependence (notebook section 8.9, validation, {len(probe_pages)} estimate pages): row-slot labels "
    "(part_1..part_8) follow the absolute height on the page. Row numbers still right after moving the page content: "
    f"10 px down {slots_right(0, 10)}; 20 px down {slots_right(0, 20)}; 40 px down {slots_right(0, 40)}; one row height "
    f"({ROW_PITCH} px) down {slots_right(0, ROW_PITCH)}; 20 px up {slots_right(0, -20)}; one row height up "
    f"{slots_right(0, -ROW_PITCH)}. Field type still right: {types_right(0, ROW_PITCH)} one row down, {types_right(40, 0)} at "
    f"40 px right, {types_right(-40, 0)} at 40 px left. Do not serve on pages not aligned with the training layout. "
    f"A lookup on the four box coordinates alone (no text, no image, no model) labels {int(baseline_scored.correct.sum()):,} of "
    f"{len(baseline_scored):,} scored validation boxes, so the validation score mostly measures the fixed layout. "
    "One M4 line box is one unit; most quantity cells are not detected by M4 and stay null; a misread quantity or amount "
    "that is a well-formed number is accepted. Model scores are uncalibrated."
    + (" QUICK RUN: plumbing check on a few pages, not a model." if QUICK_RUN else ""))
print("known_limits of the entry:\n ", KNOWN_LIMITS, "\n")
TARGETS = {"m5_complete_entry_f1": {
    "target": "at least 0.75 on held-out page and template variants within the supported layout families (M5 specification)",
    "status": "not_evaluated",
    "reason": "validation scores on one synthetic layout; complete-entry scoring on the frozen project test cases was not run"}}
CREATED_BY = "Lane 2 (document extraction), notebooks/M5_Document_Worker.ipynb" + (" [QUICK RUN]" if QUICK_RUN else "")

scratch_registry = Path(tempfile.mkdtemp(prefix="serving-check-", dir=EVALUATION_DIR))
check_entry = export_for_serving(RUN_DIR, f"lineitems/{RUN_ID}-check", scratch_registry, check_pages=validation_pages[:4],
                                 created_by=CREATED_BY, known_limits=KNOWN_LIMITS, targets=TARGETS)
print("Scratch entry:", shown(check_entry), "| problems:", entry_problems(check_entry) or "none")
print("Files:", sorted(p.name for p in check_entry.iterdir()))
print("preprocessing.json:", json.dumps({k: v for k, v in json.loads((check_entry / "preprocessing.json").read_text()).items()
                                         if k in serving_preprocessing("line")}))
served = LocalLayoutLMv3(check_entry, str(device))
print("Loaded by LocalLayoutLMv3 | text unit:", served.text_unit, "| model version:", served.model_version[:23] + "…")

served_predictions = {p.image_id: (served.predict(p.page, p.rectified_path) if p.page_status != "unreadable" else ())
                      for p in validation_pages}
compared = [(a, b) for p in validation_pages for a, b in zip(served_predictions[p.image_id], predictions[p.image_id])]
same_labels = sum(a.box_id == b.box_id and a.label == b.label for a, b in compared)
largest_gap = max(abs(a.confidence - b.confidence) for a, b in compared)
print(f"Serving loader against the notebook's model: {share(same_labels, len(compared))} box labels identical; "
      f"largest confidence difference {largest_gap:.2e}")
assert same_labels == len(compared) == sum(len(predictions[p.image_id]) for p in validation_pages), "serving and notebook disagree"
shutil.rmtree(scratch_registry)      # the scratch copy of the weights is not needed once the model is in memory

**What to look for.** Read the `known_limits` text once: it is what a later reader of the registry will see first. `problems: none` means the entry has every file the training specification requires and no silently missing manifest field. The comparison line must say that **all** box labels are identical: the application's loader, reading the exported files one page at a time, gives the same answer as the notebook's batched inference. The tiny confidence difference is floating-point noise between a batch of eight and a batch of one.

### 10.1 From labels to `LineItem` records

`adapt_predictions` is the part of the application that turns box labels into records. For every row slot it collects the boxes labelled `part_N_description`, `…_action`, `…_quantity`, `…_amount`; maps the description and the operation through the reviewed vocabulary; parses the amount as an **exact decimal** (a string like `7,03Q.00` is rejected, not repaired); and writes a `LineItem` with links to the source boxes. Currency, cost basis and the layout family's number format are explicit inputs: nothing is inferred. Remarks and page fields stay *unparsed evidence*.

The cell runs it claim by claim (both pages, as the application would) and compares each line item with the claim's ground truth:

- **printed amount**: exact decimal equality with `line_total`;
- **quantity** and **operation**: equality with the claim record;
- **description**: text similarity (0 to 1) between the OCR text and the printed description;
- **vocabulary mapping**: whether the description resolved to a part code.

In [ ]:
family_config, vocabulary = load_layout_families(), load_estimate_vocabulary()
FAMILY_ID, CURRENCY, COST_BASIS = "family-a-ruled-grid", "SGD", "single_part_pre_tax_no_discount_v1"   # explicit inputs
provenance = Provenance(source_kind="synthetic", runtime_profile="full", producer_service="notebook-m5-layoutlmv3")
by_claim = defaultdict(list)
for page in validation_pages:
    by_claim[page.dataset_claim_id].append(page)

item_rows, results, completeness_states = [], {}, Counter()
for claim_key, claim_pages in by_claim.items():
    claim_id = claim_pages[0].claim_id
    job_key = make_job_key(claim_id, 1, "line_items_extract", {"line_items_model": served.model_version,
                                                               "extraction_method": "layoutlmv3"}, target="document")
    result = adapt_predictions([p.page for p in claim_pages], {p.page.page_id: served_predictions[p.image_id] for p in claim_pages},
                               config=family_config, vocabulary=vocabulary, family_id=FAMILY_ID, job_key=job_key,
                               claim_id=claim_id, input_revision=1, currency=CURRENCY, cost_basis=COST_BASIS,
                               model_version=served.model_version, provenance=provenance, min_confidence=MIN_CONFIDENCE)
    results[claim_key] = result
    completeness_states[result.completeness.state] += 1
    estimate = next(p for p in claim_pages if p.page_type == "repair_estimate")
    item_rows += line_item_rows(result, estimate.page.page_id, evaluation["alignments"][estimate.image_id],
                                annotations[estimate.image_id], claims[claim_key])
items = pd.DataFrame(item_rows)
summary = line_item_summary(item_rows)
(EVALUATION_DIR / "line_items_validation.json").write_text(json.dumps(
    {"split": "validation", "provenance": "synthetic", "quick_run": QUICK_RUN, **summary}, indent=2))
total = summary["true_rows"]
clean, inked = summary["amount_exact_clean"], summary["amount_exact_ink_over"]
print("SYNTHETIC VALIDATION EVIDENCE" + (" · QUICK RUN, not a model result" if QUICK_RUN else ""))
print(f"Claims: {len(by_claim)} | declaration completeness: {dict(completeness_states)}")
print(f"Ground-truth rows: {total} ({summary['rows_on_unreadable_pages']} on pages M4 could not read)")
print("Line items produced for a true row :", share(summary["line_items_found"], total), "| extra line items:", summary["extra_line_items"])
print("Printed amount exact (Decimal)     :", share(summary["amount_exact"], total))
print("   rows with a clean amount        :", share(clean["amount_exact"], clean["rows"]))
print("   rows with ink over the amount   :", share(inked["amount_exact"], inked["rows"]))
print("Quantity equals the claim record   :", share(summary["quantity_match"], total))
print("Operation equals the claim record  :", share(summary["operation_match"], total))
print("Description mapped to a part code  :", share(summary["part_mapped"], total))
def amount_outcome(row):
    if not row["found"]:
        return "4 no line item (model, OCR or unreadable page)"
    if row["amount_exact"]:
        return "1 exact"
    return "3 no amount accepted (null, with a reason)" if pd.isna(row["amount"]) else "2 WRONG number accepted"

true_rows = items[~items.extra.astype(bool)].copy()
true_rows["outcome"] = true_rows.apply(amount_outcome, axis=1)
true_rows["amount box"] = np.where(true_rows.ink_over_amount.fillna(False).astype(bool), "ink over the amount", "clean")
outcome_table = pd.crosstab(true_rows["outcome"], true_rows["amount box"], margins=True, margins_name="all rows")
print("\nWhat happened to the printed amount of every ground-truth row:")
display(outcome_table)
wrong_accepted = true_rows[true_rows.outcome.str.startswith("2")]
if len(wrong_accepted):
    print(f"Rows where a wrong number was accepted ({len(wrong_accepted)}; the first 8):")
    display(wrong_accepted[["image_id", "slot", "row_style", "amount_text", "amount", "true_amount"]].head(8))
else:
    print("No row with a wrong accepted number on these pages.")
def quantity_outcome(row):
    if not row["found"]:
        return "4 no line item (model, OCR or unreadable page)"
    if row["quantity_match"]:
        return "1 right"
    return "3 no quantity accepted (null, with a reason)" if pd.isna(row["quantity"]) else "2 WRONG value accepted"

true_rows["quantity outcome"] = true_rows.apply(quantity_outcome, axis=1)
true_rows["quantity box"] = np.where(true_rows.quantity_delivered.astype(bool), "M4 delivered a box", "no box from M4")
print("\nWhat happened to the quantity of every ground-truth row:")
display(pd.crosstab(true_rows["quantity outcome"], true_rows["quantity box"], margins=True, margins_name="all rows"))
not_right = true_rows[true_rows.quantity_delivered.astype(bool) & ~true_rows.quantity_match.astype(bool)]
print(f"Rows where M4 delivered a quantity box and the record does not carry the true quantity ({len(not_right)}):")
if len(not_right):
    display(not_right[["image_id", "slot", "quantity_text", "quantity_ocr_confidence", "quantity", "true_quantity", "quantity_reason"]])
similarity = summary["description_similarity_mean"]
print("Mean description similarity        :", "no line item to compare" if similarity is None else
      f"{similarity:.3f} over {summary['description_similarity_rows']} line items")

**How to read these numbers.**

- **Printed amount exact** is the number that matters most downstream. Compare the *clean* line with the *ink over the amount* line, then read the outcome table: each ground-truth row ends in exactly one of four outcomes.
  - `1 exact`: the record carries the true printed amount.
  - `2 WRONG number accepted`: **the dangerous case.** The text M4 read under the ink was a different but perfectly well-formed number (`415.00` read as `150`), so the exact parser had no reason to refuse it. The rows are listed under the table. Neither OCR confidence nor the parser catches these; the pen mark on the row (M6) and the surveyor do.
  - `3 no amount accepted`: the parser refused the text (`7,03Q.00`, `-410.00-`) or a confidence was too low. The amount is null with a reason, which is the safe failure.
  - `4 no line item`: nothing was produced for the row, for example on a page M4 could not read.
- **Quantity** has its own outcome table, with the same four outcomes, and it is the weakest field. Read the columns first. `no box from M4` is the finding of section 4.2: OCR did not detect the digit, there is nothing to label, and the quantity stays **null** with the reason `value_missing`. It is never set to one. `M4 delivered a box` is the small remainder, and even there not every quantity is right. In the reference run, of 206 rows:
  - 21 are right;
  - **3 are wrong values that were accepted.** OCR read the printed `1` as `7` (OCR confidence 0.59 to 0.73). `7` is a perfectly good quantity, so the record carries it with no `field_uncertainty` at all (SYN-00195 row 6, SYN-00204 row 5, SYN-00206 row 1);
  - 168 are null: 164 without a box, 3 withheld because the model's confidence in the label was below 0.5 although OCR had read `1` correctly (the three boxes of section 8.6), and 1 refused by the parser because OCR read the letter `I`;
  - 14 have no line item.
- **The lesson of both outcome tables: a well-formed wrong value passes every format check.** `7` is a valid quantity. `8` is a valid amount: on SYN-00178 row 4 OCR read `0008` under an amount cross, the parser accepted it as `8`, and the page prints `130.00`. The exact-decimal parser checks the *form* of a value, and a confidence threshold checks a score. Neither can see that a clean number is the wrong number. Only a **cross-check** can: the totals printed on the page, a pen mark on the row (M6), or a person. The adapter used here compares no row with the printed totals, so today the guard is M6 and the surveyor.
- **Mapped to a part code** is reported as it is. The reviewed vocabulary (`configs/pipeline/m5_estimate_vocabulary.yaml`) does not cover many of this dataset's descriptions and abbreviations. Aliases are team-reviewed, so this notebook does not edit them; an unmapped description stays visible with its original text.
- **Declaration completeness is `partial` for every claim, by design.** The model has not shown that it finds *every* row of a table (omission recall) or where a table ends, and one synthetic layout cannot show it. So the adapter never reports `complete`, and M8 will not treat an unmatched damage as a confirmed missing repair. Zero rows on a readable page would also be `partial`, never "explicitly empty": only a human can state that.

The next cell shows what the records look like for one claim, next to the page they came from.

In [ ]:
found_items = items[(~items.extra) & items.found]
if found_items.empty:
    print("No line item was produced on the validation pages, so there is nothing to show."
          + (" That is normal for a reduced-data reload check." if QUICK_RUN else ""))
else:
    inked_items = found_items[found_items.ink_over_amount.fillna(False).astype(bool)]
    show_id = (inked_items if len(inked_items) else found_items).image_id.iloc[0]
    show_page = cache.by_id(show_id)
    by_id = {b.box_id: b for b in evaluation["alignments"][show_id].boxes}
    shown_rows = scored[(scored.image_id == show_id) & (scored.pred != "O")]
    bottom = max(by_id[box_id].box_px[3] for box_id in shown_rows[shown_rows.pred_slot.notna()].box_id) + 40
    fig, ax = plt.subplots(figsize=(13, 13 * (bottom - 400) / 1160))
    draw_boxes(ax, Image.open(show_page.rectified_path).convert("RGB"), [by_id[r.box_id].box_px for r in shown_rows.itertuples()],
               [TYPE_COLOURS[r.pred_type] for r in shown_rows.itertuples()], texts=[r.pred[2:] for r in shown_rows.itertuples()],
               text_size=6.5, linewidth=1.8, extent=(40, 400, 1200, bottom))
    ax.set_title(f"{show_id} (validation): the model's field labels on the parts table")
    show(fig, "20_line_items_example")
    display(items[items.image_id == show_id][["slot", "row_style", "ink_over_amount", "description", "part_code", "operation",
                                             "quantity", "amount_text", "amount", "true_amount", "amount_exact"]])
    item = results[show_page.dataset_claim_id].line_items[0]
    print("One LineItem record (shortened):")
    print(json.dumps({k: v for k, v in item.model_dump(mode="json").items() if k in (
        "entry_id", "original_part_text", "part_code", "part_mapping_status", "side", "operation", "quantity",
        "printed_line_amount", "effective_price", "effective_price_source", "currency", "cost_basis", "field_uncertainty",
        "source_box_ids", "versions")}, indent=1, default=str)[:1500])

**What to look for.** In the table, `amount_text` is what OCR read, `amount` is what the exact parser accepted (empty when it refused), `true_amount` is the claim record. If a `description` or an `amount_text` ever holds two values at once, two different rows were given the same slot and the adapter joined them: a slot error damages two rows, not one. A row where `amount_text` is not a clean number has `amount` empty and a `field_uncertainty` entry: the system shows the doubt and does not guess. In the record, `quantity` is null with a reason, `source_box_ids` link every value to the M4 boxes it came from, and `versions` names the exact weights that produced it.

#### The rows M6 receives

M6 detects the surveyor's pen marks, and then has to say **which row** each mark belongs to. For that it is given, for every line item, a *row box* and an *amount box* (the `row_boxes` of the command `cmev.cmd.pen-marks-detect.v1`). The application builds them with `pen_mark_row_boxes`, printed below: four lines.

The next cell writes these boxes for the validation estimate pages to `rows_for_m6_validation.json` in the evaluation folder, so that the M6 notebook can link marks against the rows M5 **really** produces, not against ideal rows taken from the ground truth. Every estimate page is listed, including the pages M4 could not read: they appear with no rows, because M6 would get none for them. The file is marked `quick_run: true` in a quick run, and M6 must not take such a file for a model result.

One thing to understand before you look at the picture: a row box is the **union of the text boxes M5 put into the row**. It is not the printed table band of the row. It is as tall as the text in it, which is much less than the band, and it grows when M5 keeps a tall box in the row.

In [ ]:
print(inspect.getsource(pen_mark_row_boxes))
rows_record = rows_for_m6(
    validation_pages, results, split="validation", quick_run=QUICK_RUN, model_version=served.model_version,
    registry_entry=registry_entry_with_weights(REGISTRY_ROOT, EXPORT_VERSION, served.model_version),   # None until it is exported
    m4_cache_manifest_sha256=sha256_file(cache.root / "manifest.json"), family_id=FAMILY_ID, min_confidence=MIN_CONFIDENCE,
    produced_by="notebooks/M5_Document_Worker.ipynb, section 10.1: LocalLayoutLMv3.predict -> adapt_predictions -> pen_mark_row_boxes")
ROWS_FOR_M6 = EVALUATION_DIR / "rows_for_m6_validation.json"
write_json_atomic(ROWS_FOR_M6, rows_record)      # written beside its target, then renamed: a reader never sees half a file
listed = rows_record["pages"]
row_boxes = [row for listed_page in listed for row in listed_page["row_boxes"]]
print(f"Wrote {shown(ROWS_FOR_M6)} (schema {rows_record['schema']}, quick_run {rows_record['quick_run']}): "
      f"{len(listed)} estimate pages, {len(row_boxes)} rows, {sum(row['amount_box_norm'] is None for row in row_boxes)} without an amount box")
print("Estimate pages with no row for M6:", [(p["image_id"], p["page_status"]) for p in listed if not p["row_boxes"]] or "none")
print("Line items on other pages, which the file does not list:", sum(len(pen_mark_row_boxes(r)) for r in results.values()) - len(row_boxes))
print("Rows are traced to the registry entry:", rows_record["m5_registry_entry"] or "none yet (set after an export in section 12)")

with_rows = [p for p in listed if p["row_boxes"]]
if not with_rows:
    print("No row was produced, so there is no picture." + (" That is normal for a reduced-data reload check." if QUICK_RUN else ""))
else:
    marked = [p for p in with_rows if annotations[p["image_id"]].get("marks")]
    drawn_page = (marked or with_rows)[0]
    drawn_cached, drawn_annotation = cache.by_id(drawn_page["image_id"]), annotations[drawn_page["image_id"]]
    page_size = np.array([drawn_cached.page.transform.corrected_width, drawn_cached.page.transform.corrected_height] * 2)
    row_px = [np.array(row["row_box_norm"]) * page_size for row in drawn_page["row_boxes"]]
    amount_px = [np.array(row["amount_box_norm"]) * page_size for row in drawn_page["row_boxes"] if row["amount_box_norm"]]
    bands = source_to_corrected_boxes(drawn_cached.page, [d["row_bbox"] for d in drawn_annotation["row_decisions"]])
    marks_px = source_to_corrected_boxes(drawn_cached.page, [m["bbox"] for m in drawn_annotation["marks"]])         if drawn_annotation.get("marks") else []
    top, bottom = min(b[1] for b in bands) - 20, max(b[3] for b in bands) + 20
    fig, ax = plt.subplots(figsize=(13, 13 * (bottom - top) / 1160))
    draw_boxes(ax, Image.open(drawn_cached.rectified_path).convert("RGB"), row_px + amount_px,
               [PALETTE["blue"]] * len(row_px) + [PALETTE["vermillion"]] * len(amount_px), linewidth=1.8, extent=(40, top, 1200, bottom))
    for x0, y0, x1, y1 in bands:
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=PALETTE["grey"], linestyle=":", linewidth=1.2))
    for x0, y0, x1, y1 in marks_px:
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor="black", linestyle="--", linewidth=1.3))
    ax.legend(handles=[Line2D([], [], color=PALETTE["blue"], lw=2, label="row box given to M6 (union of the row's M5 boxes)"),
                       Line2D([], [], color=PALETTE["vermillion"], lw=2, label="amount box given to M6"),
                       Line2D([], [], color=PALETTE["grey"], lw=1.2, ls=":", label="printed table band of the row (ground truth, not given to M6)"),
                       Line2D([], [], color="black", lw=1.3, ls="--", label="surveyor pen mark (ground truth; M6 must detect it)")],
              loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=2, fontsize=8)
    ax.set_title(f"{drawn_page['image_id']} (validation): the {len(row_px)} row boxes and amount boxes that M6 receives from M5")
    show(fig, "21_rows_for_m6")
    heights = np.array([(row["row_box_norm"][3] - row["row_box_norm"][1]) * page_size[1] for row in row_boxes])
    band_height = float(np.median([b[3] - b[1] for b in bands]))
    print(f"Height of the {len(heights)} row boxes: median {np.median(heights):.0f} px, from {heights.min():.0f} to {heights.max():.0f} px; "
          f"the printed band of a row on this page is {band_height:.0f} px high")

**What to look for.** The dotted grey bands are the printed table rows. The blue boxes are what M6 is actually given: each hugs the text M5 assigned to the row, so it covers only part of the band's height and usually not its full width (no blue box reaches the quantity column when OCR missed the quantity). The red box is the row's amount box. The black dashes are the surveyor's marks, which M6 has to find and then attach to one of the blue boxes. A mark drawn in the empty part of a band, away from the text, overlaps its blue box only a little; that is why the row-box definition matters to M6's linking.

In the reference run the file lists 38 estimate pages and 192 rows, every one with an amount box. Two pages (`SYN-00201_P2`, `SYN-00213_P2`) have no rows because M4 could not read them, so M6 can link nothing on them. The row boxes are 24 to 50 pixels high (median 31), in bands of 79 pixels. How well M6 links marks with these rows is measured in the M6 notebook, not here.

### 10.2 A comparison with the earlier checkpoint

`artifacts/models/document-worker-layoutlmv3` is the checkpoint from `notebooks/Document_Worker.ipynb`. It is useful to see, on the **same validation M4 pages**, what training on M4 output changes.

**The caveat, stated before the numbers.** That model was trained on the dataset's annotation regions: clean text, exact boxes, field values only. It was never shown a form label, a column header, handwriting or an OCR mistake, and no training example was `O`. On M4 output about half of the boxes are exactly those things. So its low score on them is a consequence of *what it was trained on*, not of the method or of the work that went into it; the present notebook only exists because that experiment showed the route was open. To run it at all, each M4 line box is given to it as one "word", which matches how it was trained (one region, full text, one box). It declares no text unit, so the serving loader treats it as a word model and refuses M4's line boxes: the first lines of the output show that refusal working as intended.

In [ ]:
if (TEAMMATE_CHECKPOINT / "model.safetensors").is_file():
    from transformers import AutoModelForTokenClassification
    try:
        LocalLayoutLMv3(TEAMMATE_CHECKPOINT).predict(validation_set.pages[0].page, validation_set.pages[0].rectified_path)
    except ValueError as refusal:
        print("Serving loader on the earlier checkpoint with M4 line boxes →", refusal, "\n")
    earlier_processor = AutoProcessor.from_pretrained(str(TEAMMATE_CHECKPOINT), apply_ocr=False, local_files_only=True)
    earlier_model = AutoModelForTokenClassification.from_pretrained(str(TEAMMATE_CHECKPOINT), local_files_only=True).to(device)
    print("Same label names as this notebook:", set(earlier_model.config.id2label.values()) == set(labels))
    earlier_predictions, _ = run_inference(earlier_model, PageWindowDataset(validation_pages, earlier_processor), device,
                                           config.eval_batch_size)
    earlier = validation_report(evaluation["alignments"], earlier_predictions, annotations)
    earlier_rows = pd.DataFrame(box_rows(evaluation["alignments"], earlier_predictions))
    def line(name, r, table):
        on_fields = table[table.scored & (table.true != "O")]
        return {"model": name, "field boxes labelled exactly right": share(int(on_fields.correct.sum()), len(on_fields)),
                "field micro precision": round(r["boxes"]["field_micro"]["precision"] or 0, 4),
                "O boxes left as O": share(r["boxes"]["outside"]["true_positive"], r["boxes"]["outside"]["support"]),
                "entity F1": round(r["entities"]["f1"], 4),
                "rows complete (delivered fields)": share(r["rows"]["complete_available"], r["rows"]["rows"])}
    comparison = pd.DataFrame([line("earlier checkpoint (trained on annotation regions)", earlier, earlier_rows),
                               line("this run (trained on M4 boxes)" + (" · QUICK RUN" if QUICK_RUN else ""), report, scored)])
    display(comparison.set_index("model").T)
    comparison.to_json(EVALUATION_DIR / "comparison_with_earlier_checkpoint.json", orient="records", indent=2)
    del earlier_model
else:
    print("No earlier checkpoint at", TEAMMATE_CHECKPOINT, "- comparison skipped.")

**How to read the comparison.** "Field boxes labelled exactly right" is the fair line for the earlier model: it looks only at boxes that are printed field values, the thing it was trained for. "O boxes left as O" is the line it could not possibly do well on. The difference between the two columns is the size of the **train/serve mismatch** this notebook was written to remove. It is not a ranking of two people's work, and both columns are synthetic validation evidence.

## 11 · Final held-out evaluation — explicit opt-in

Leave `FINAL_EVAL = False` while anything is still being chosen: the label policy, the recipe, the checkpoint, a threshold. The test split (74 pages, 37 claims) is the only data this notebook has not looked at. The first time its scores influence a decision, it stops being a test set. When everything is frozen, set `FINAL_EVAL = True` and run once. The cell uses the same `evaluate_pages` as section 8.2 and writes `test_metrics.json` under the evaluation folder.

"Once" is enforced, not just asked for. Before a single test page is read, `open_final_evaluation` creates `test_metrics.json` for this run. It **refuses**:

- in a quick run, because a plumbing check must not spend the test split;
- when `test_metrics.json` already exists for this run, whether it holds a finished report or marks an opening that did not finish. The error names the file. Read that file instead of scoring again: a test split that is scored a second time, after its first result was seen, is no longer held out.

Even then the result is **synthetic** test evidence: same layout, same fonts. It supports the statement "the model generalises to unseen claims of this generator", and nothing wider.

In [ ]:
if FINAL_EVAL:
    # Refuses a quick run and a second opening, and reserves the report file before any test page is read.
    test_report_path = open_final_evaluation(EVALUATION_DIR, run_id=RUN_ID, quick_run=QUICK_RUN)
    test_cache = load_m4_cache(CACHE_DIR, DATASET_DIR, splits=("test",))
    test_pages = list(test_cache.split("test"))
    test_annotations = read_annotations(DATASET_DIR, splits=("test",))
    final = evaluate_pages(model, processor, test_pages, test_annotations, POLICY, labels, device=device, batch_size=config.eval_batch_size)
    final_report = final["report"]
    write_json_atomic(test_report_path, {"split": "test", "provenance": "synthetic", "run_id": RUN_ID, "status": "completed",
                                         "quick_run": QUICK_RUN, **final_report})
    print("SYNTHETIC TEST EVIDENCE · run", RUN_ID, "· opened once, after selection on validation")
    print(f"Pages: {final_report['pages']} ({final_report['pages_unreadable']} unreadable for M4)")
    display(pd.DataFrame([{"score": "field labels, micro (boxes)", **final_report["boxes"]["field_micro"]},
                          {"score": "O, reported apart", **final_report["boxes"]["outside"]},
                          {"score": "entities", **{k: final_report["entities"][k] for k in final_report["boxes"]["field_micro"]}}]).round(4))
    display(pd.DataFrame(final_report["per_type"]).round(4))
    print("Rows:", final_report["rows"])
else:
    print("Final test remains unused. Continue with validation only.")

## 12 · Export for serving — explicit opt-in

Set `EXPORT_FOR_SERVING = True` to write the run as a registry entry under `REGISTRY_ROOT/lineitems/<run_id>-serving/`. To export a run a second time, for example after its limits were measured better, set `EXPORT_NAME` to a new name such as `<run_id>-serving-r2`: the suffix says that it replaces an earlier export of the same weights, and the earlier folder is then retired by a person, not by this notebook. The export:

- stages the entry in a temporary folder, checks it, and only then moves it into place;
- **refuses** if the destination exists: a registry entry is written once and never changed;
- writes the weights and processor in the layout that the adapter loads on transformers 4 and 5, plus `manifest.json`, `label_schema.json`, `preprocessing.json`, `split_manifest.json`, `evaluation_report.json`, `training_log.jsonl`, `config.yaml` and `NOTICE.md`;
- records status `candidate`, provenance `synthetic`, the M4 cache manifest hash and the M4 versions the model was trained on;
- loads the staged entry with `LocalLayoutLMv3` and requires it to reproduce the run's labels on real validation pages;
- afterwards writes the entry's name into `rows_for_m6_validation.json`, so the rows M6 reads can be traced to a registry entry.

Exporting does **not** serve the model. The neural M5 path is not wired into a worker yet (`lineitems.method = layoutlmv3` is a specified switch, not an implemented one), so the cell prints what a human would record and the command to try the candidate by hand.

In [ ]:
print("Entry name:", EXPORT_VERSION, "(set in section 2; an existing entry is never replaced)")
if EXPORT_FOR_SERVING:
    entry = export_for_serving(RUN_DIR, EXPORT_VERSION, REGISTRY_ROOT, check_pages=validation_pages[:8],
                               created_by=CREATED_BY, known_limits=KNOWN_LIMITS, targets=TARGETS)
    exported = json.loads((entry / "manifest.json").read_text())
    print("Candidate written:", entry)
    print("Problems found by the handover check:", entry_problems(entry) or "none")
    print("\nRecord these values wherever the candidate is referred to (copy exactly):\n")
    print(json.dumps({"model_id": exported["model_id"], "model_version": exported["version"], "status": exported["status"],
                      "weights_sha256": exported["weights_sha256"], "text_unit_granularity": exported["text_unit_granularity"],
                      "label_schema_version": exported["label_schema_version"],
                      "preprocessing_version": exported["preprocessing_version"],
                      "label_alignment_policy": exported["conversion_version"]["label_alignment_policy"]["version"],
                      "m4_cache_manifest_sha256": exported["m4_cache"]["manifest_sha256"],
                      "m4_ocr_weights": exported["m4_cache"]["versions"].get("ocr_weights"),
                      "provenance": exported["provenance"]}, indent=2))
    print("\nTry it by hand on M4 pages (see docs/layoutlmv3_line_item_extraction.md for the input file):\n")
    print(f"python scripts/extract_line_items_layoutlmv3.py <input.json> --output <line-items.json> --model-dir {entry} "
          f"--family-id {FAMILY_ID} --currency {CURRENCY} --cost-basis {COST_BASIS} --device {device.type}")
    if "sha256:" + exported["weights_sha256"] == rows_record["m5_model_version"]:      # the same weights the rows came from
        rows_record["m5_registry_entry"] = EXPORT_VERSION
        write_json_atomic(ROWS_FOR_M6, rows_record)
        print(f"\n{shown(ROWS_FOR_M6)} now names this entry as the source of its rows.")
    if QUICK_RUN:
        print("\nQUICK RUN export: a plumbing check. Do not use this entry as a model.")
else:
    print("Nothing exported. Set EXPORT_FOR_SERVING = True to write", EXPORT_VERSION)

## 13 · Interpretation, limits, glossary and next steps

### Reference run of 2026-10-12

Numbers of the run `lineitems-layoutlmv3-20261012T143930Z-ec8bcba-s20260922` (default recipe, Apple MPS, 14 epochs, stopped early, epoch 9 selected, 22 minutes). They are **synthetic validation evidence** on 76 pages of 38 claims. Your own run prints its own numbers above; a repeat on a GPU differs slightly.

**Where these numbers can be checked.** The reference copy of this notebook with all its outputs is `artifacts/evaluation/notebook-validation/M5_Document_Worker-reload-20261013.ipynb` (with an `.html` rendering beside it). It is this revision of the notebook, executed on 2026-10-13 with `RELOAD_RUN_DIR` pointing at the run above and `EXPORT_FOR_SERVING = True`. The model was trained on 2026-10-12 by an earlier revision of this notebook, with the same training code (`train()` in `pipelines/documents/lineitems_training.py`). In this correction pass both quick and full executions **reload saved weights only**. The training branch was skipped and no optimization step was run, as requested. The run's registry candidate is `lineitems/lineitems-layoutlmv3-20261012T143930Z-ec8bcba-s20260922-serving-r2`. It holds the same weights as the first export of 2026-10-12 (`…-serving`), which was retired because its `known_limits` were written before the position dependence was measured.

| What | Measured | Section |
| --- | --- | --- |
| Validation pages M4 could not read | 2 of 76 (kept as failures in every row count) | 3.1 |
| Text boxes on the 74 readable pages | 4,356: 4,324 scored, 32 masked as ambiguous | 4.2 |
| Scored boxes labelled correctly | 4,324 of 4,324 (2,128 field boxes and 2,196 `O` boxes) | 8.2 |
| Entities correct | 2,128 of 2,128. No validation field has two boxes, so this repeats the row above. Another 301 ground-truth fields had no usable M4 box | 8.2 |
| Position lookup with no model (nearest training box) | 4,285 of 4,324 scored boxes: 2,089 of 2,128 field boxes and 2,196 of 2,196 `O` boxes | 8.2 |
| Quantity cells M4 delivered | 28 of 206 rows (training: 150 of 976) | 4.2, 8.5 |
| Quantity cells OCR read exactly | 24 of the 28 delivered (training: 131 of 150); the misreads are `7` or `I` for a printed `1` | 4.4 |
| Rows completely labelled, delivered fields | 192 of 206. The other 14 are on the 2 unreadable pages | 8.5 |
| Rows completely labelled, all four fields | 28 of 206: M4 delivered all four fields for only 28 rows | 8.5 |
| Correct boxes below the 0.5 confidence threshold | 3 of 4,324, all quantity cells; no box was wrong | 8.6 |
| Handwritten boxes labelled as a printed amount | 0 of 70 (and 0 of the 32 merged boxes) | 8.7 |
| Merged print-and-handwriting boxes | 32: the model calls 31 `O` and 1 a printed remark | 8.7 |
| Row slot right after moving the page **down** | 716 of 716 cells at 10 px, 633 at 20 px, 311 at 40 px, 19 at 79 px (one row; 697 are one row too high), 13 at 158 px | 8.9 |
| Row slot right after moving the page **up** | 716 of 716 cells at 10 px, 683 at 20 px, 253 at 40 px, 162 at 79 px (one row; 554 are one row too low) | 8.9 |
| Field type right under the vertical moves | 716 of 716 at every move down; 716, 715, 715 and 716 at 10, 20, 40 and 79 px up | 8.9 |
| Moved sideways (field type and row slot give the same count) | right: 716, 716 and 694 of 716 at 10, 20 and 40 px; left: 716, 713 and 688. Every lost cell is a quantity cell labelled `O` | 8.9 |
| Serving loader gives the notebook's labels | 4,356 of 4,356 boxes | 10 |
| Line items produced for a ground-truth row | 192 of 206 | 10.1 |
| Printed amount exact | 177 of 206: clean amounts 129 of 138, ink over the amount 48 of 68 | 10.1 |
| Wrong number accepted as the printed amount | 12 of 206, all with ink over the amount (for example OCR text `0008` accepted as `8`, true `130.00`) | 10.1 |
| Quantity of the 206 rows | 21 right; 3 wrong values accepted (`7` for a printed `1`); 168 null (164 no box, 3 withheld by the 0.5 threshold, 1 refused by the parser); 14 no line item | 10.1 |
| Rows handed to M6 | 192 rows on 38 estimate pages, each with an amount box; 2 of the pages have no rows (M4 could not read them) | 10.1 |
| Operation equal to the claim record | 189 of 206 | 10.1 |
| Description mapped to a part code | 90 of 206 | 10.1 |
| Earlier checkpoint, same pages: field boxes right | 1,994 of 2,128, and 0 of 2,196 `O` boxes left as `O` | 10.2 |

### What the numbers show

- A LayoutLMv3 model **trained on M4's own output** labels the text boxes of *this* synthetic layout, including the boxes the earlier checkpoint never saw (form labels, headers, handwriting as `O`).
- **The task is easy on this generator.** A lookup on box coordinates alone, with no model, labels 4,285 of the 4,324 scored boxes. The model's perfect score is 39 boxes better than that. Read every box score in this notebook against this baseline.
- Treating **one M4 line box as one unit** is workable on this layout, where a table cell is one OCR line: boxes that had to be masked as ambiguous are rare and are counted in section 4.2. This is the "measured justification" that `docs/document-neural-workflow.md` asked for, **limited to this layout**.
- The exported entry is loadable by the application's serving adapter and reproduces the notebook's labels exactly (section 10).
- **Labelling is not the weak link on these pages; the input is.** The amounts, quantities and operations that are lost in section 10.1 are lost before the labelling step: pages M4 refused, quantity cells OCR did not detect, and text misread under ink. The low share of descriptions mapped to a part code is a separate matter: it measures how much of this dataset's wording the reviewed vocabulary covers.

### What the numbers do not show

- **Nothing about real estimates.** One page layout, one generator, three handwriting *fonts*, flat images with no perspective, blur or shadow. Train, validation and test differ only in the claim values. A perfect validation score is the expected result of an easy, self-similar task.
- **Nothing about a page that is positioned differently, and section 8.9 measured that it fails there.** The row-slot labels let the model learn "row 3 is at this height". The model tolerates a move of about 10 pixels. From a quarter of a row height on, up or down, row numbers go wrong, and at a whole row height nearly every cell is renumbered by one, while field types stay right. A real scan or photograph will not hold the table within ten pixels of the training position.
- **Nothing about a field in several boxes.** No validation field is carried by two boxes, so the entity score repeats the box score here, and how the model labels a wrapped description is unmeasured.
- **Not the M5 acceptance target.** The specification's complete-entry F1 (part and side, operation, quantity, currency and basis, amount, on the frozen project test cases) was not evaluated. A row here cannot be a complete entry when its quantity never reached M5.
- **Not reading accuracy.** M5 labels boxes; M4 reads them. Section 4.4 and section 10.1 show printed amounts under ink are often misread, sometimes with high OCR confidence, and that the few quantity cells M4 delivers are its worst-read field. That part of the error is outside this model. And a misread that is a well-formed number is accepted: nothing in this pipeline yet compares a row with the printed totals.
- **Not calibrated confidence**, and the test split is unopened.

### Findings to hand on (for the document lane)

0. **The trained model takes the row number from the absolute height on the page** (section 8.9). It tolerates a move of about 10 pixels and breaks at a fraction of a row height, up or down. It must not be served on pages that are not aligned with the training layout until that is fixed and re-measured with the same probe.
1. **M4 misses most quantity cells** on this layout (section 4.2). It is a text-detection question for M4. M5 leaves a missing quantity null. **And M4 misreads some of the quantity cells it does find**: 4 of 28 on validation. Three of them, a printed `1` read as `7`, were accepted as the quantity of a line item with no uncertainty flag (section 10.1). A null quantity is the safe outcome; a wrong accepted quantity is not.
2. **Three pages are refused by M4** (nested OCR regions in handwriting). They are failures in every denominator here; M5 has nothing to read on them.
3. **Print merged with handwriting** in one OCR line happens in the remarks column. Those boxes are masked in training and stay ambiguous in serving.
4. **OCR confidence does not flag misread amounts under ink**, and a misread amount can be a well-formed number that the exact parser accepts (section 10.1): 12 of 206 validation rows, for example OCR text `0008` accepted as the amount `8` where the page prints `130.00` (SYN-00178 row 4). **A well-formed wrong value passes every format check.** Only a cross-check can catch it: the printed totals, a pen mark on the row (M6), or a person. A confidence threshold cannot.
5. **The 0.5 model-confidence threshold has no evidence for it yet.** On validation it caught no wrong box, because there was none, and it withheld 3 quantities that were labelled and read correctly (section 8.6).
6. **The validation score mostly measures the fixed layout**: a position lookup with no model reaches 4,285 of 4,324 boxes (section 8.2).
7. **What M6 gets from M5** is in `rows_for_m6_validation.json` (section 10.1): 192 row boxes on 38 pages, none for the two pages M4 refused. A row box is the union of the row's text boxes, 24 to 50 pixels high inside a printed band of 79 pixels, not the band itself. Which row box M5 should supply to M6 is a question for the two lanes together.
8. The label policy, its thresholds, the recipe and the 0.5 confidence threshold are **proposed** and need a team decision.

### Next steps toward a served M5

- **Make row numbering independent of the position on the page.** This comes first, because section 8.9 shows the present model depends on it. Two candidate routes, both to be judged with the same probe: (a) train with random shifts and scales of the whole page, moving the image and the box coordinates together, so the coordinates stay exact; (b) switch to **field-type labels plus geometric row grouping**: predict only `description`/`amount`/… and group cells into rows by their vertical position. Route (b) also removes the limit of eight rows.
- **Real pages.** Collect permitted photographed estimates of the supported layouts, run M4 on them, and repeat sections 4 and 8 on them as a separate denominator. Expect to label by hand: there is no generator ground truth.
- **A second layout**, as a held-out template, to measure what survives a change of form.
- **Word boxes or line boxes?** Keep the region-level unit if cells stay one line on real pages; otherwise evaluate a word-localising OCR configuration. The adapter already supports both and refuses a mismatch.
- **Wire the worker**: a Kafka consumer for `cmev.cmd.line-items-extract.v1` behind the explicit `lineitems.method` switch, loading a registry entry only in the `full` profile, with the rule-based parser kept as the default.
- **M6** consumes the row and amount boxes of these line items to link pen marks to rows; a price-change mark on a row is the signal that a printed amount needs a human. The M6 notebook reads `rows_for_m6_validation.json` from this run to measure that link with real M5 rows.
- **A cross-check for well-formed wrong values**: compare each row's quantity and amount with the totals printed on the page, as the rule-based parser's subtotal check does, and flag a row that does not add up. It is the only kind of guard that can catch a misread `7` or `8`.
- Open the **test split once**, after the choices above are frozen.

### Glossary

| Term | Meaning here |
| --- | --- |
| Text box | One rectangle of text found by OCR, with the string read and a confidence |
| Granularity | Whether a text box is a word, a line or a block. M4 delivers lines |
| Corrected page | M4's deskewed or rectified page image; the frame of all boxes |
| `box_norm` / `bbox` | A box as 0–1 fractions of the corrected page / as 0–999 integers for LayoutLMv3 |
| Token, subword | A piece of a word from the model's fixed vocabulary; the model's unit of text |
| First-subword rule | A box's label is trained and read at its first token only |
| `-100`, masked | A position the loss ignores: padding, later subwords, ambiguous boxes |
| BIO | `B-x` begins field `x`, `I-x` continues it, `O` is outside any field |
| Row slot | The `N` in `part_N_<field>`: the Nth printed row of the table on a page |
| Logit | The model's raw score for one label at one position, before softmax |
| Softmax | Turns the logits of one position into probabilities: exponentiate each, divide by their sum |
| Cross-entropy | The loss: minus the log of the probability given to the true label |
| Epoch | One pass over all training pages |
| AdamW | The optimiser: it turns gradients into weight changes, with a step size of its own for every weight |
| Weight decay | A small shrinking of every weight at each update, which discourages very large weights |
| Gradient clipping | Scaling the gradient down when its length exceeds a limit (here 1), so one batch cannot move the weights far |
| Warmup | The first updates, during which the learning rate rises from near zero |
| Precision / recall / F1 | Share of predictions that are right / share of true items found / their harmonic mean |
| Micro / macro | Averaged over boxes / averaged over labels |
| Entity | A field together with exactly the boxes that carry it |
| Calibration | Whether a confidence of 0.8 means "right 80 % of the time" |
| Expected calibration error | The gap between confidence and share correct, averaged over confidence groups weighted by their size |
| Baseline | The score of a deliberately simple method (here: a lookup by position) that a model's score is read against |
| Homography | A 3 × 3 matrix that maps points of one flat view of a page to another, perspective tilt included |
| CER | Character error rate: edit distance divided by the length of the true text |
| Candidate | A trained and scored model that the lane owner has not accepted |
| Synthetic | Generated data. Evidence from it is never evidence about real documents |

The last cell lists where this run left its files.

In [ ]:
print("Run folder       :", RUN_DIR)
print("Evaluation folder:", EVALUATION_DIR)
print("Figures saved    :", len(list(FIGURES_DIR.glob("*.png"))))
print("Reports          :", sorted(p.name for p in EVALUATION_DIR.glob("*.json")))
if QUICK_RUN:
    print("QUICK RUN: plumbing check only. No number in this notebook run is a model result.")